# Per-Night Polarized Source Filtering

**by Tyler Cox and Josh Dillon**, last updated November 13, 2026

This notebook fits polarized point source models (position + rotation measure) from a night of HERA
single-baseline data, then builds Faraday-rotating DPSS models suitable for subtracting those sources
from the visibilities. For each source in the catalog, it:

1. Loads pseudo-Stokes Q and U visibilities over a narrow LST window around each source.
2. Iteratively fits the source's sky position and rotation measure to the complex linear polarization pQ + i pU.
3. Forms a baseline-averaged, phase-rotated dataset for all sources.
4. Subtracts off a set of sources whose sidelobes intersect the target sources.
5. Fits a separable DPSS model (in time and frequency) to the Faraday-rotated signal.

With pQ = ee - nn and pU = en + ne, the complex linear polarization pQ + i pU carries the source's polarized
flux in both Q and U, and holds a Faraday-rotating source at a single Faraday depth, +RM; pQ alone holds it
at both +RM and -RM, with half the amplitude at each. Setting `FIT_PSEUDO_STOKES_U = False` fits pQ alone.

A polarized source with fewer than `MIN_TIMES` usable integrations (any channel in `FIT_FREQ_RANGE` unflagged)
in its LST window is not modeled, so its whole window is flagged instead.

The bright sources of step 4 are subtracted whenever a polarized source with `use_sidelobe_model` is modeled.
Each one's position is fit around its own transit if at least `MIN_TIMES` usable integrations fall in its LST
window, and taken from the catalog otherwise.

Unlike its H6C IDR3.2 predecessor, it works on flagged, *non-inpainted* single-baseline files,
combined with the night's sky-filtered flag waterfall: every fit and filter treats flagged cells as
zero weight, so no inpainted values are ever relied on. That lets it run before the inpainting stage.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths and wrapper-level toggles.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: Faraday Depth Spectra](#Figure-1:-Faraday-Depth-Spectra)

• [Figure 2: Source Images and Fit Positions](#Figure-2:-Source-Images-and-Fit-Positions)

• [Figure 3: Prior Source Images and Fit Positions](#Figure-3:-Prior-Source-Images-and-Fit-Positions)

• [Figure 4: Polarized Source Model vs. Data (Low Band)](#Figure-4:-Polarized-Source-Model-vs.-Data-(Low-Band))

• [Figure 5: Polarized Source Model vs. Data (High Band)](#Figure-5:-Polarized-Source-Model-vs.-Data-(High-Band))

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import re
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import glob
import warnings
from multiprocessing import Pool

import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy import constants
from astropy.coordinates import SkyCoord
from astropy.stats import mad_std
from scipy import optimize
from tqdm.notebook import tqdm

from pyuvdata import Telescope, UVBeam, UVData
from hera_cal import frf, io, redcal, utils, polfilt, datacontainer, flag_utils
from hera_cal.smooth_cal import _linear_fit
from hera_filters import dspec
import uvtools
import snapshot_imager

import yaml
from pyuvdata import UVFlag

from IPython.display import display
%matplotlib inline

## Load Files for Polarized Source Modeling

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "PER_NIGHT_SOURCE_FILTERING_NOTEBOOK")

# default names for the files this notebook reads or writes: single-baseline files end in the suffix, per-night
# products are relative to SUM_FILE's folder ({JD} is the integer JD), and the source catalogs are absolute paths
SINGLE_BASELINE_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'
FLAGS_SKY_FILTERED_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_sky_filtered.h5'
SOURCE_CATALOG_FILENAME = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/source_catalogs/sources_to_filter.yaml'  # an absolute path
PRIOR_SOURCE_CATALOG_FILENAME = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/source_catalogs/prior_sources.yaml'  # likewise; bright sources to subtract before modeling the polarized ones
SOURCE_MODELS_FILENAME = 'single_baseline_files/phased.{JD}.{source}.{model}.uvh5'
FLAGS_UNMODELED_SOURCES_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_unmodeled_sources.h5'

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)

# default settings, overridden by the TOML's [PER_NIGHT_SOURCE_FILTERING_OPTS] section (if given)
MIN_TIMES = 20  # minimum number of usable integrations (any channel in FIT_FREQ_RANGE unflagged) in a source's LST window to model it, or to fit a prior source's position; with fewer, a polarized source's whole window is flagged and a prior source keeps its catalog position
FIT_WINDOW_TIMES = 50  # integrations in each source's position and rotation-measure fit window
FIT_FREQ_RANGE = [107.0, 230.0]  # in MHz; the range used to fit source positions and rotation measures
# fit source positions and rotation measures to the complex linear polarization pQ + i * PSEUDO_STOKES_U_SIGN * pU,
# with pQ = ee - nn and pU = en + ne (else to pQ alone)
FIT_PSEUDO_STOKES_U = True
# +1 when the E and N feeds give pQ = -Q and pU = +U (IAU), so that pQ + i pU = -(Q - iU) appears at +RM under
# hera_cal.polfilt's de-rotation by exp(2i lambda^2 RM); -1 when one feed's polarity is reversed
PSEUDO_STOKES_U_SIGN = 1
MAX_RM_DIFF = 2.0  # in rad / m^2; fits farther than these from the catalog fall back to the catalog values
MAX_RA_DIFF = 1.0  # in degrees
MAX_DEC_DIFF = 1.0  # in degrees
MIN_SNR = 5.0  # of the fit source in its image, likewise
FG_DELAY_HALF_WIDTH = 750.0  # in ns; the foreground DPSS filter applied before modeling each source
SIDELOBE_DELAY_HALF_WIDTH = 100.0  # in ns; the low-pass filter that turns a phased average into a prior source's model
MODEL_DELAY_HALF_WIDTH = 150.0  # in ns; the source model's spectral DPSS half-width
MODEL_FRATE_HALF_WIDTH = 0.5  # in mHz; the source model's temporal DPSS half-width
EIGENVAL_CUTOFF = 1e-12
# fit each Faraday-rotating source with physical RM modes selected from the unweighted source/foreground geometry,
# residualizing every integration's basis with the same weights as its foreground filter (else a two-branch fit)
RESIDUALIZED_SOURCE_BASIS = True
# evaluate the fit on the physical RM modes (the source itself), not on their residualized (foreground-filtered) response,
# so the source's part inside the foreground filter's span is removed too (it is moved out of fg_model)
PHYSICAL_SOURCE_MODEL = True
# keep only RM modes with at most this fraction f of their power in the foreground subspace, measured without flags
# (must be < 1); with PHYSICAL_SOURCE_MODEL, a kept mode's part inside that subspace is then at most sqrt(f / (1 - f)) of
# the part the fit measures (1/3 at 0.1) in an unflagged integration, and flags can raise that ratio
SOURCE_MAX_FOREGROUND_FRACTION = 0.1
SOURCE_BASIS_RCOND = 1e-12  # numerical rank cutoff for orthonormalizing the physical source subspace
FIT_MAXITER = 10  # maximum source-parameter fitting iterations
MAX_WORKERS = 8  # parallel workers for loading files

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'PER_NIGHT_SOURCE_FILTERING_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val
assert 0 <= SOURCE_MAX_FOREGROUND_FRACTION < 1, f'SOURCE_MAX_FOREGROUND_FRACTION must be in [0, 1), not {SOURCE_MAX_FOREGROUND_FRACTION}'
assert PSEUDO_STOKES_U_SIGN in (1, -1), f'PSEUDO_STOKES_U_SIGN must be 1 or -1, not {PSEUDO_STOKES_U_SIGN}'

def settings_note():
    '''The settings above as Python assignments. Pasted into a cell right after this one, the printed note reproduces a run
    interactively, since everything derived from the settings is computed after it.'''
    return '\n'.join(f'{setting} = {eval(setting)!r}' for setting in [
        'SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SINGLE_BASELINE_SUFFIX', 'CORNER_TURN_MAP_FILENAME',
        'FLAGS_SKY_FILTERED_FILENAME', 'SOURCE_CATALOG_FILENAME', 'PRIOR_SOURCE_CATALOG_FILENAME', 'SOURCE_MODELS_FILENAME',
        'FLAGS_UNMODELED_SOURCES_FILENAME', 'BAND_SPLIT_FREQ', 'MIN_TIMES', 'FIT_WINDOW_TIMES', 'FIT_FREQ_RANGE', 'FIT_PSEUDO_STOKES_U', 'PSEUDO_STOKES_U_SIGN', 'MAX_RM_DIFF', 'MAX_RA_DIFF', 'MAX_DEC_DIFF', 'MIN_SNR', 'FG_DELAY_HALF_WIDTH',
        'SIDELOBE_DELAY_HALF_WIDTH', 'MODEL_DELAY_HALF_WIDTH', 'MODEL_FRATE_HALF_WIDTH', 'EIGENVAL_CUTOFF', 'RESIDUALIZED_SOURCE_BASIS', 'PHYSICAL_SOURCE_MODEL',
        'SOURCE_MAX_FOREGROUND_FRACTION', 'SOURCE_BASIS_RCOND', 'FIT_MAXITER', 'MAX_WORKERS'])

print(settings_note())

In [ ]:
# copy-paste settings note here to interactively override the above defaults or whatever comes from the TOML

In [ ]:
# everything derived from the settings, computed after them so that settings pasted in to reproduce a run carry through
jdstr = re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)
CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
FLAGS_SKY_FILTERED_FILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_SKY_FILTERED_FILENAME.format(JD=jdstr))
FLAGS_UNMODELED_SOURCES_FILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_UNMODELED_SOURCES_FILENAME.format(JD=jdstr))
# the pseudo-Stokes visibilities loaded for the polarized sources, and the polarization their positions and rotation
# measures are fit to, pQ + i * U_SIGN * pU (pQ alone when U_SIGN is 0)
FIT_POLS = ('pQ', 'pU') if FIT_PSEUDO_STOKES_U else ('pQ',)
U_SIGN = PSEUDO_STOKES_U_SIGN if FIT_PSEUDO_STOKES_U else 0
FIT_POL_LABEL = {0: 'pQ', 1: 'pQ + i pU', -1: 'pQ - i pU'}[U_SIGN]
for name in ['CORNER_TURN_MAP_YAML', 'FLAGS_SKY_FILTERED_FILE', 'FLAGS_UNMODELED_SOURCES_FILE', 'FIT_POLS', 'FIT_POL_LABEL']:
    print(f'{name} = {eval(name)}')

In [ ]:
add_to_history = ('Produced by per_night_source_filtering notebook with the following settings:\n' + settings_note() + '\n'
                  + 'and the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65)

## Preliminaries

In [ ]:
# the night's single-baseline files, from the corner-turn map; autocorrelations (one per polarization) are kept apart
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)
all_outfiles = sorted({outfile for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles})
baselines = [tuple(map(int, os.path.basename(f).split('.')[3].split('_'))) for f in all_outfiles]
auto_file = [f for (ai, aj), f in zip(baselines, all_outfiles) if ai == aj]
load_files = [f for (ai, aj), f in zip(baselines, all_outfiles) if ai != aj and os.path.exists(f)]
print(f'Found {len(load_files)} cross-correlation single-baseline files (and {len(auto_file)} autocorrelation files) '
      f'in {os.path.dirname(CORNER_TURN_MAP_YAML)}.')

# the night's flag waterfall from the sky-filtered round of RFI flagging: the single-baseline files carry only the flags
# the calibration left, so this is what marks the RFI, and it is OR'd into every read below
uvf = UVFlag(FLAGS_SKY_FILTERED_FILE)
prior_flags = np.all(uvf.flag_array, axis=-1)
hd_meta = io.HERAData(load_files[0])
if FIT_PSEUDO_STOKES_U:
    assert {'en', 'ne'} <= set(hd_meta.pols), \
        f'{load_files[0]} has polarizations {hd_meta.pols}, but fitting pseudo-Stokes U needs en and ne (or set FIT_PSEUDO_STOKES_U = False).'
assert prior_flags.shape == (len(hd_meta.times), len(hd_meta.freqs)), \
    f'{FLAGS_SKY_FILTERED_FILE} has shape {prior_flags.shape}, but the single-baseline files have {len(hd_meta.times)} times and {len(hd_meta.freqs)} channels.'
assert np.allclose(uvf.time_array, hd_meta.times) and np.allclose(uvf.freq_array, hd_meta.freqs), \
    f'{FLAGS_SKY_FILTERED_FILE} is not on the same time and frequency grid as the single-baseline files.'
print(f'Sky-filtered flags: {np.mean(prior_flags):.3%} of the waterfall flagged.')

In [ ]:
# the channels used to fit source positions and rotation measures (the high band, by default), and the integrations with
# all of them flagged, which neither count toward a source's LST window nor anchor its fit window
FREQ_SLICE = np.where((hd_meta.freqs >= FIT_FREQ_RANGE[0] * 1e6) & (hd_meta.freqs <= FIT_FREQ_RANGE[1] * 1e6))[0]
combined_time_flags = np.all(prior_flags[:, FREQ_SLICE], axis=1)
print(f'{np.mean(combined_time_flags):.3%} of integrations are entirely flagged from {FIT_FREQ_RANGE[0]} to {FIT_FREQ_RANGE[1]} MHz.')

## Identify Which Sources Are Within Range

In [ ]:
# the polarized sources to model, and (optionally) the bright sources to subtract before modeling them
with open(SOURCE_CATALOG_FILENAME) as f:
    source_list = yaml.safe_load(f)["sources"]
with open(PRIOR_SOURCE_CATALOG_FILENAME) as f:
    prior_source_list = (yaml.safe_load(f) or {}).get("sources") or []  # a present-but-empty `sources:` is an empty catalog

# Find the polarized sources with at least MIN_TIMES usable integrations within lst_half_width of transit to fit, so that each source's
# fit window is anchored on a usable integration inside its LST window. A polarized source with fewer is neither modeled
# nor subtracted, so its whole LST window is flagged instead: otherwise this one night would keep the source there
sources = {}
unmodeled_source_flags = np.zeros_like(prior_flags)

print ("Polarized sources chosen for model:")
for si, source in enumerate(source_list):
    source_name = source["name"]
    source_coord = SkyCoord(ra=source["right_ascension"], dec=source["declination"])
    
    lst_center = source_coord.ra.deg / 15 * np.pi / 12
    in_window = (
        (hd_meta.lsts > lst_center - source["lst_half_width"] * np.pi / 12) &
        (hd_meta.lsts < lst_center + source["lst_half_width"] * np.pi / 12)
    )
    lst_mask = in_window & ~combined_time_flags
    if lst_mask.sum() >= MIN_TIMES:
        sources[source_name] = {
            "right_ascension":  source_coord.ra.deg,
            "declination":      source_coord.dec.deg,
            "rotation_measure": source["rotation_measure"],
            "fit_scintillation": source["fit_scintillation"],
            "scintillation_timescale": source["scintillation_timescale"],
            "lst_half_width": source["lst_half_width"] * np.pi / 12,
            "use_sidelobe_model": source["use_sidelobe_model"]
        }
        print(f'\t{source_name}: {lst_mask.sum()} usable integrations in window -- INCLUDED')
    else:
        unmodeled_source_flags[in_window] = True
        print(f'\t{source_name}: {lst_mask.sum()} usable integrations in window -- SKIPPED (fewer than {MIN_TIMES})'
              + (f'; flagging all {in_window.sum()} of its integrations' if np.any(in_window) else ''))

# The prior sources matter only through their sidelobes in the LST windows of the polarized sources that use sidelobe
# models, which is where their models are built and subtracted. So every one is subtracted whenever such a source is
# modeled, whether or not its own transit has usable data. Its position is fit around that transit when its own LST
# window has at least MIN_TIMES usable integrations (so that the fit window is anchored inside it), and taken from the
# catalog otherwise.
prior_sources = {}

print ("\nSources chosen for subtraction prior to modeling polarized sources:")
if not prior_source_list:
    print("\tNo prior sources found; skipping prior-source subtraction.")
elif not any(source["use_sidelobe_model"] for source in sources.values()):
    print("\tNo modeled polarized source uses sidelobe models; skipping prior-source subtraction.")
else:
    for source in prior_source_list:
        source_name = source["name"]
        source_coord = SkyCoord(ra=source["right_ascension"], dec=source["declination"])

        lst_center = source_coord.ra.deg / 15 * np.pi / 12
        lst_mask = (
            (hd_meta.lsts > lst_center - source["lst_half_width"] * np.pi / 12) &
            (hd_meta.lsts < lst_center + source["lst_half_width"] * np.pi / 12) &
            ~combined_time_flags
        )
        prior_sources[source_name] = {
            "right_ascension":  source_coord.ra.deg,
            "declination":      source_coord.dec.deg,
            "lst_half_width": source["lst_half_width"] * np.pi / 12,
            "fit_position": lst_mask.sum() >= MIN_TIMES,
        }
        print(f'\t{source_name}: {lst_mask.sum()} usable integrations in its own window -- INCLUDED, '
              + ('position fit there' if prior_sources[source_name]["fit_position"]
                 else f'at its catalog position (fewer than {MIN_TIMES} to fit it)'))

## Load Pseudo-Stokes Q and U Data Around Each Source

In [ ]:
def closest_values(v, p, k=FIT_WINDOW_TIMES):
    """Return ``(start, end)`` indices of the *k*-element window in *v* centred
    nearest to value *p*.

    This is the flag-unaware fallback used when no unflagged integrations exist.
    Prefer ``closest_unflagged_values`` at all normal call sites.

    Parameters
    ----------
    v : array_like
        Sorted 1-D array to search.
    p : float
        Target value.
    k : int
        Window length (default FIT_WINDOW_TIMES).

    Returns
    -------
    start, end : int
        Slice indices such that ``v[start:end]`` is the closest *k*-element
        segment to *p*.
    """
    idx = np.searchsorted(v, p)
    half = k // 2
    start = max(0, idx - half)
    end = start + k
    if end > len(v):
        end = len(v)
        start = end - k
    return start, end


def closest_unflagged_values(v, p, time_flags, k=FIT_WINDOW_TIMES):
    """Return (start, end) of a k-element window centred on the nearest unflagged
    integration to p.

    Finds the single unflagged integration in v closest to p and centres a
    standard k-wide window on it.  The window always contains exactly k
    indices (fewer only when v is shorter than k; near the edges of v the window
    is shifted inward, off-centre).  Some of those k integrations may
    themselves be flagged — that is fine, they will be downweighted during
    fitting.  The key improvement over ``closest_values`` is that the window
    anchor is the best unflagged time near the peak rather than the bare LST
    centre, which matters when flagging is concentrated around transit.

    Parameters
    ----------
    v : array_like
        Sorted 1-D array (e.g. LST timestamps in radians).
    p : float
        Target value to centre on (source peak LST).
    time_flags : array_like of bool
        Per-integration flag array; True = flagged/unusable.  Should be
        ``combined_time_flags`` (integrations with every channel in FIT_FREQ_RANGE flagged in the sky-filtered flags).
    k : int
        Number of integrations to load (default FIT_WINDOW_TIMES).

    Returns
    -------
    start, end : int
        Contiguous slice indices such that ``v[start:end]`` has length k
        (or len(v) when k > len(v)).  Falls back to ``closest_values``
        when no unflagged integrations exist.
    """
    v          = np.asarray(v)
    time_flags = np.asarray(time_flags, dtype=bool)

    unflagged_inds = np.where(~time_flags)[0]

    if len(unflagged_inds) == 0:
        warnings.warn(
            f"No unflagged integrations near LST={p:.4f} rad; "
            "falling back to closest_values."
        )
        return closest_values(v, p, k)

    # Anchor on the single unflagged integration nearest to the source peak.
    best_idx = int(unflagged_inds[np.argmin(np.abs(v[unflagged_inds] - p))])

    # Build a standard k-wide window centred on that anchor.
    half  = k // 2
    start = max(0, best_idx - half)
    end   = start + k
    if end > len(v):
        end   = len(v)
        start = max(0, end - k)

    return start, end


def has_usable_data(flags_dc, nsamples_dc, prior_flags_slice):
    """Return True if at least one unflagged, non-zero-weight sample exists.

    Combines the DataContainer flags with the prior flag slice and checks
    whether any weight is positive.  Used as a fast pre-check before calling
    expensive fitting routines so that all-flagged sources are skipped
    gracefully rather than producing a degenerate fit.

    Parameters
    ----------
    flags_dc : DataContainer
        Per-baseline boolean flag arrays, shape (n_times, n_freqs).
    nsamples_dc : DataContainer
        Per-baseline sample-weight arrays.
    prior_flags_slice : ndarray, shape (n_times, n_freqs)
        Prior flag array sliced to the same time range as the DataContainers.

    Returns
    -------
    bool
        True when at least one sample is unflagged and has positive weight.
    """
    for key in nsamples_dc:
        combined = flags_dc[key] | prior_flags_slice
        if np.any(nsamples_dc[key][~combined] > 0):
            return True
    return False

In [ ]:
# the instrumental polarizations each pseudo-Stokes visibility is formed from, with their signs
PSEUDO_STOKES_COMBINATIONS = {
    "pI": {"ee": 1, "nn": 1},
    "pQ": {"ee": 1, "nn": -1},
    "pU": {"en": 1, "ne": 1},
}


def _process_file_hera(args):
    """Read a single HERA file and extract pseudo-Stokes visibilities for every source.

    For each source defined in the passed ``sources`` dict the function:

    1. Reads the instrumental polarizations that ``pols`` needs (``ee`` and ``nn``
       for pI and pQ, ``en`` and ``ne`` for pU) over a narrow LST window centred
       on the source's right ascension.
    2. Forms each pseudo-Stokes visibility in ``pols`` for every baseline:
       ``pI = ee + nn``, ``pQ = ee - nn`` and ``pU = en + ne``.
    3. Gives all of a baseline's pseudo-Stokes visibilities the same flags (the
       OR of the flags of every polarization read and the night's sky-filtered
       flags) and the same sample weights, ``(sum of 1 / nsamples)^-1`` over the
       polarizations read (non-finite values are zeroed out).

    Parameters
    ----------
    args : tuple of (str, dict, tuple of str)
        A tuple of (filepath, sources, pols) where filepath is the path to a
        single-baseline HERA UVH5 data file, sources is a dict of source
        parameters keyed by source name, and pols names the pseudo-Stokes
        visibilities to form, keys of ``PSEUDO_STOKES_COMBINATIONS``.

    Returns
    -------
    out_data, out_flags, out_nsamples : tuple of lists
        Three parallel lists (one entry per source) of baseline-keyed dicts
        containing the pseudo-Stokes visibilities, combined flags, and sample
        weights respectively; a source whose read fails is left empty.
    """
    filepath, sources, pols = args  # Unpack since Pool.imap requires a single argument
    read_pols = sorted({p for pol in pols for p in PSEUDO_STOKES_COMBINATIONS[pol]})

    hd = io.HERAData(filepath)

    out_data     = [{} for _ in sources]
    out_flags    = [{} for _ in sources]
    out_nsamples = [{} for _ in sources]

    for si, source in enumerate(sources):
        lst_center = sources[source]["right_ascension"] / 15 * np.pi / 12
        imin, imax = closest_unflagged_values(
            hd.lsts, lst_center, combined_time_flags, k=FIT_WINDOW_TIMES
        )
        lst_range  = (hd.lsts[imin], hd.lsts[imax - 1])

        try:
            d, f, n = hd.read(
                polarizations=read_pols,
                lst_range=lst_range,
                freq_chans=FREQ_SLICE,
            )
        except Exception:
            continue

        for ap in d.antpairs():
            # shared by all of this baseline's pseudo-Stokes visibilities, so that they can be combined
            flags = np.logical_or.reduce([f[ap + (p,)] for p in read_pols] + [prior_flags[imin:imax][:, FREQ_SLICE]])
            with np.errstate(divide="ignore", invalid="ignore"):
                nv = np.sum([n[ap + (p,)] ** -1.0 for p in read_pols], axis=0) ** -1.0
            nv = np.where(np.isfinite(nv), nv, 0.0)

            for pol in pols:
                out_data[si][ap + (pol,)] = sum(sign * d[ap + (p,)] for p, sign in PSEUDO_STOKES_COMBINATIONS[pol].items())
                out_flags[si][ap + (pol,)] = flags
                out_nsamples[si][ap + (pol,)] = nv

    return out_data, out_flags, out_nsamples


def load_parallel(files, sources, pols=("pQ",), max_workers=16):
    """Load and merge HERA data across multiple files in parallel.

    Files are processed concurrently using :class:`multiprocessing.Pool`.
    After all workers finish, metadata (antenna positions, frequencies, JD
    times) is attached by re-reading the first file serially.

    Parameters
    ----------
    files : list of str
        Ordered list of file paths to load.  The first entry is used as the
        metadata reference.
    sources : dict
        Dictionary of source parameters keyed by source name.
    pols : tuple of str
        The pseudo-Stokes visibilities to form, keys of
        ``PSEUDO_STOKES_COMBINATIONS``. All of a baseline's share its flags
        and sample weights.
    max_workers : int
        Number of worker processes in the pool (default: 16).

    Returns
    -------
    data_list, flags_list, nsamples_list : dict of DataContainer
        Pseudo-Stokes visibilities (one key per baseline and polarization in
        ``pols``), combined flags, and sample weights, each keyed by source name.

    Raises
    ------
    ValueError
        If ``pols`` names a visibility not in ``PSEUDO_STOKES_COMBINATIONS``.
    """
    unknown = set(pols) - set(PSEUDO_STOKES_COMBINATIONS)
    if unknown:
        raise ValueError(f"Cannot form {sorted(unknown)}; pols must be among {sorted(PSEUDO_STOKES_COMBINATIONS)}.")

    data     = [{} for _ in sources]
    flags    = [{} for _ in sources]
    nsamples = [{} for _ in sources]

    args = [(f, sources, tuple(pols)) for f in files]  # Bundle sources with each filepath

    with Pool(processes=max_workers) as pool:
        results = pool.imap_unordered(_process_file_hera, args, chunksize=1)
        for result in tqdm(results, total=len(files)):
            if result is None:
                continue
            try:
                out_data, out_flags, out_nsamples = result
                for si in range(len(sources)):
                    data[si].update(out_data[si])
                    flags[si].update(out_flags[si])
                    nsamples[si].update(out_nsamples[si])
            except Exception as exc:
                print(f"Merge error: {exc}")
                raise

    # Attach metadata from the first file and wrap in DataContainers
    hd = io.HERAData(files[0])
    data_list     = {}
    flags_list    = {}
    nsamples_list = {}

    for si, source_name in enumerate(sources):
        lst_center = sources[source_name]["right_ascension"] / 15 * np.pi / 12
        imin, imax = closest_unflagged_values(
            hd.lsts, lst_center, combined_time_flags, k=FIT_WINDOW_TIMES
        )
        lst_range  = (hd.lsts[imin], hd.lsts[imax - 1])

        try:
            d, _, _ = hd.read(
                polarizations=["ee", "nn"],
                lst_range=lst_range,
                freq_chans=FREQ_SLICE,
            )
        except Exception:
            continue

        dc = datacontainer.DataContainer(data[si])
        dc.antpos = d.antpos
        dc.freqs  = d.freqs
        dc.times  = d.times

        data_list[source_name]     = dc
        flags_list[source_name]    = datacontainer.DataContainer(flags[si])
        nsamples_list[source_name] = datacontainer.DataContainer(nsamples[si])

    return data_list, flags_list, nsamples_list

In [ ]:
data_list, flags_list, nsamples_list = load_parallel(
    load_files, sources, pols=FIT_POLS, max_workers=MAX_WORKERS
)

In [ ]:
# pseudo-Stokes-I data around the transits of the prior sources whose positions are fit
prior_sources_to_fit = {name: source for name, source in prior_sources.items() if source["fit_position"]}
if prior_sources_to_fit:
    prior_data_list, prior_flags_list, prior_nsamples_list = load_parallel(
        load_files, prior_sources_to_fit, pols=('pI',), max_workers=MAX_WORKERS
    )
else:
    # Avoid launching workers and rereading every file when no prior source's
    # position is fit.
    prior_data_list = {}
    prior_flags_list = {}
    prior_nsamples_list = {}

## Fit Source Positions and Rotation Measures

In [ ]:
def unpack_polarized_visibilities(data, flags, nsamples, pol="pQ", u_sign=0):
    """Unpack pseudo-Stokes visibilities into arrays, as ``hera_cal.utils.unpack_data_containers``.

    With ``u_sign`` 0 this is ``utils.unpack_data_containers(data, flags, nsamples, pol=pol)``. With ``u_sign`` +1
    or -1, ``vis`` is instead the complex linear polarization ``pQ + 1j * u_sign * pU`` (``pol`` is ignored). Its
    sky brightness is complex, so its visibilities are not Hermitian: a baseline's conjugate row is
    ``conj(pQ) + 1j * u_sign * conj(pU)``, not the conjugate of the baseline's own row. It is therefore formed from
    the pQ and pU arrays after each is unpacked with its conjugate rows.

    Parameters
    ----------
    data, flags, nsamples : DataContainer
        Pseudo-Stokes visibilities, flags and sample weights. With nonzero ``u_sign`` they must hold pQ and pU with
        the same flags and weights, as ``load_parallel`` gives them.
    pol : str
        The pseudo-Stokes visibility to unpack when ``u_sign`` is 0.
    u_sign : int
        The coefficient of ``1j * pU`` in the complex linear polarization: +1 or -1, or 0 for ``pol`` alone.

    Returns
    -------
    vis, weights, uvw, times, freqs
        As ``hera_cal.utils.unpack_data_containers``: a row for each baseline and one for its conjugate, so ``vis``
        and ``weights`` have shape (2 * n_antpairs, n_times, n_freqs) and ``uvw`` (2 * n_antpairs, 3, n_freqs).

    Raises
    ------
    ValueError
        If ``u_sign`` is not -1, 0 or 1, or pQ and pU differ in their baselines or weights.
    """
    if u_sign == 0:
        return utils.unpack_data_containers(data, flags, nsamples, pol=pol)
    if u_sign not in (1, -1):
        raise ValueError(f"u_sign must be -1, 0 or 1, not {u_sign}.")
    vis, weights, uvw, times, freqs = utils.unpack_data_containers(data, flags, nsamples, pol="pQ")
    vis_u, weights_u, uvw_u, _, _ = utils.unpack_data_containers(data, flags, nsamples, pol="pU")
    if not (np.array_equal(uvw, uvw_u) and np.array_equal(weights, weights_u)):
        raise ValueError("pQ and pU must share baselines, flags and weights to be combined.")
    vis += 1j * u_sign * vis_u
    return vis, weights, uvw, times, freqs


def fit_polarized_source_params(data, flags, nsamples, right_ascension, declination, rotation_measure, location,
                                u_sign=0, maxiter=10, drm=5.0, dtest=5000, verbose=False):
    """Iteratively fit the RA, Dec and rotation measure of a polarized source.

    Alternates hera_cal.polfilt's position fit and coherent rotation-measure search, as
    ``polfilt.iteratively_fit_polarized_source_params`` does, on the visibilities ``unpack_polarized_visibilities``
    gives: pQ when ``u_sign`` is 0, else the complex linear polarization pQ + i * u_sign * pU. Each search covers
    ``drm`` either side of the previous iteration's RM. The fit has converged when RA and Dec change by less than
    1e-4 degrees and RM by less than 1e-3 rad/m^2 in one iteration.

    Parameters
    ----------
    data, flags, nsamples : DataContainer
        Pseudo-Stokes visibilities, flags and sample weights over the source's fit window.
    right_ascension, declination : float
        Initial position in degrees.
    rotation_measure : float
        Initial rotation measure in rad/m^2.
    location : astropy.coordinates.EarthLocation
        Telescope location.
    u_sign : int
        See ``unpack_polarized_visibilities``.
    maxiter : int
        Maximum number of position/RM iterations.
    drm, dtest : float, int
        Half-width (rad/m^2) and number of trial values of each RM search.
    verbose : bool
        Print each iteration's estimate.

    Returns
    -------
    fit_ra, fit_dec, fit_rm : float
        The last estimate (the initial values if every sample has zero weight).
    converged : bool
        Whether the fit converged within ``maxiter`` iterations.
    """
    vis, weights, uvw, times, freqs = unpack_polarized_visibilities(data, flags, nsamples, u_sign=u_sign)
    weights = np.where(np.isfinite(vis), weights, 0.0)
    vis = np.where(np.isfinite(vis), vis, 0.0)
    if np.sum(weights) == 0.0:
        return right_ascension, declination, rotation_measure, False

    # polfilt's per-step fits take arrays; its iterative wrapper unpacks a single polarization, which cannot be pQ + i pU
    ra, dec, rm = right_ascension, declination, rotation_measure
    for iteration in range(maxiter):
        fit_ra, fit_dec = polfilt._fit_polarized_source_position(vis, weights, uvw, ra, dec, rm, times, freqs, location)
        fit_rm = polfilt._fit_rotation_measure(vis, weights, uvw, times, freqs, fit_ra, fit_dec, rm, location,
                                               drm=drm, dtest=dtest)
        if verbose:
            print(f"Iteration {iteration}: RA = {fit_ra}, Dec = {fit_dec}, RM = {fit_rm}")
        converged = abs(fit_ra - ra) < 1e-4 and abs(fit_dec - dec) < 1e-4 and abs(fit_rm - rm) < 1e-3
        ra, dec, rm = fit_ra, fit_dec, fit_rm
        if converged:
            return ra, dec, rm, True
    return ra, dec, rm, False

In [ ]:
"""
Diagnostics of the fit polarized source parameters (RA, Dec, Rotation Measure).

The fit itself is fit_polarized_source_params, above; these functions evaluate the
Faraday depth spectrum around a source and image it.
"""
def phased_spectrum(data, flags, nsamples, ra, dec, location, pol="pQ", u_sign=0):
    """
    Weighted average over baselines and times of the visibilities phased to ``(ra, dec)``.

    This is the spectrum whose Faraday depth ``hera_cal.polfilt._fit_rotation_measure`` searches.

    Parameters
    ----------
    data, flags, nsamples : datacontainer.DataContainer
        Pseudo-Stokes visibilities; see ``unpack_polarized_visibilities`` for ``pol`` and ``u_sign``.
    ra, dec : float
        Phase centre in degrees.
    location : astropy.coordinates.EarthLocation
        Observatory location.

    Returns
    -------
    freqs : np.ndarray, shape (n_freqs,)
        Frequencies in Hz.
    spectrum : np.ndarray, shape (n_freqs,)
        The phased, averaged visibilities; zero in channels with no weight.
    """
    vis, weights, uvw, times, freqs = unpack_polarized_visibilities(
        data, flags, nsamples, pol=pol, u_sign=u_sign
    )
    weights = np.where(np.isfinite(vis), weights, 0.0)
    vis = np.where(np.isfinite(vis), vis, 0.0)

    l, m, n = polfilt.radec_to_lmn(ra, dec, times, location)
    lmn = np.array([l, m, n])  # shape (3, n_times)
    phasor = np.exp(-2j * np.pi * np.einsum("bcf,ct->btf", uvw, lmn))

    weight_sum = np.sum(weights, axis=(0, 1))  # (n_freqs,)
    safe_weight_sum = np.where(weight_sum > 0, weight_sum, 1.0)
    spectrum = np.sum(vis * phasor * weights, axis=(0, 1)) / safe_weight_sum
    return freqs, spectrum


def faraday_response(spectrum, freqs, test_rm):
    """
    Amplitude of ``spectrum`` de-rotated by each trial rotation measure.

    ``|mean over channels of spectrum * exp(2i lambda^2 RM)|``, the quantity
    ``hera_cal.polfilt._fit_rotation_measure`` maximizes; channels with no
    weight count as zeros in the mean.

    Parameters
    ----------
    spectrum : np.ndarray, shape (n_freqs,)
        From ``phased_spectrum``.
    freqs : np.ndarray, shape (n_freqs,)
        Frequencies in Hz.
    test_rm : array_like
        Trial rotation measures in rad/m^2.

    Returns
    -------
    np.ndarray, shape (len(test_rm),)
    """
    lambda_sq = (constants.c.value / freqs) ** 2
    return np.array([
        np.abs(np.nanmean(spectrum * np.exp(2j * lambda_sq * rm)))
        for rm in np.atleast_1d(test_rm)
    ])

def image_polarized_data(data, flags, nsamples, ra, dec, rotation_measure, location, pol="pQ", u_sign=0, npix=300, fov=20, verbose=False):
    """Form a dirty image of the polarized sky around a source.

    Phase-rotates all baselines to the source position, applies the Faraday
    phasor ``exp(2i lambda^2 RM)`` to de-rotate in frequency, and accumulates a
    weighted 2-D DFT image on a flat-sky ``(l, m)`` grid.

    Parameters
    ----------
    data, flags, nsamples : DataContainer
        Pseudo-Stokes visibilities (keyed by ``pol``), flags, and weights for the source window.
    ra, dec : float
        Phase-centre right ascension and declination in degrees.
    rotation_measure : float
        Source rotation measure in rad/m^2.
    location : EarthLocation
        Telescope location used by ``polfilt.radec_to_lmn``.
    pol : str
        ``"pQ"`` (default) or ``"pI"``, the polarization key of the visibilities in ``data``.
    u_sign : int
        If +1 or -1, image the complex linear polarization pQ + i * u_sign * pU
        instead of ``pol``; see ``unpack_polarized_visibilities``.
    npix, fov : int, float
        Image size in pixels and field of view, passed to ``snapshot_imager.snapshot_imager_type1``.
    verbose : bool
        Passed to ``snapshot_imager.snapshot_imager_type1``.

    Returns
    -------
    images : the result of ``snapshot_imager.snapshot_imager_type1``
        Its ``images`` attribute holds the dirty image of the time-averaged
        data (callers ``np.squeeze`` it to 2-D).
    """
    if u_sign == 0:
        imaging_data = snapshot_imager.unpack_data_containers(
            data, flags, nsamples, pol=pol
        )
    else:
        # every conjugate baseline stored explicitly (ImagingData's default layout), since pQ + i pU is not Hermitian
        vis, weights, uvw, times, freqs = unpack_polarized_visibilities(data, flags, nsamples, u_sign=u_sign)
        imaging_data = snapshot_imager.ImagingData(vis=vis, weights=weights, uvw=uvw, times=times, freqs=freqs)
    weights = np.where(np.isfinite(imaging_data.vis), imaging_data.weights, 0.0)
    vis     = np.where(np.isfinite(imaging_data.vis), imaging_data.vis,     0.0)
    
    # Get phasors
    lambda_sq = (constants.c.value / imaging_data.freqs) ** 2
    rm_phasor = np.exp(2j * lambda_sq * rotation_measure)  # (n_freqs,)
    l0, m0, n0 = polfilt.radec_to_lmn(ra, dec, imaging_data.times, location)
    lmn = np.array([l0, m0, n0])       # (3, n_times)
    phase0 = np.einsum("bcf,ct->btf", imaging_data.uvw, lmn)                      # (n_bls, n_times, n_freqs)
    
    vis_phased = (
        vis
        * np.exp(-2j * np.pi * phase0) * weights
    )
    vis_avg = np.mean(vis_phased, axis=1, keepdims=True)
    w_avg = np.mean(weights, axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):  # cells with no weight are 0 / 0, zeroed below
        vis_avg /= w_avg
    vis_avg = np.where(np.isfinite(vis_avg), vis_avg, 0.0)

    # Put in format for snapshot imager
    imaging_data.vis = vis_avg
    imaging_data.weights = w_avg
    imaging_data.times = np.mean(imaging_data.times)
    images = snapshot_imager.snapshot_imager_type1(
        imaging_data, 
        fov=fov, 
        npix=npix, 
        rm_phasor=rm_phasor,
        verbose=verbose
    )
    return images

In [ ]:
def report_fit_checks(source_name, checks, fit_description):
    '''Print whether a source's fit is kept and why, and return the names of the checks it fails.

    Each check is (name, value, format, condition, threshold, unit): with condition '>', the value must exceed the
    threshold; with 'within', its magnitude must be below it. The fit is kept if it passes every check. Prints the
    verdict, one row per check (its value, what it must be, and pass or FAIL), then ``fit_description``.'''
    rows, failed = [], []
    for name, value, value_format, condition, threshold, unit in checks:
        passed = value > threshold if condition == '>' else abs(value) < threshold
        requirement = f'> {threshold:g}' if condition == '>' else f'within ±{threshold:g} {unit}'.strip()
        value_text = f'{value:{value_format}} {unit}'.strip()
        rows.append(f'    {name:<14}{value_text:>18}   must be {requirement:<20}{"pass" if passed else "FAIL"}')
        if not passed:
            failed.append(name)
    print(f'{source_name}: fit KEPT' if not failed
          else f'{source_name}: fit REJECTED (fails {", ".join(failed)}); catalog values used')
    print('\n'.join(rows))
    print(f'    {fit_description}\n')
    return failed


def fit_title(source_name, snr, failed):
    '''Figure title saying whether a source's fit was kept, with the fit's SNR.'''
    verdict = 'fit kept' if not failed else f'catalog values used: fit fails {", ".join(failed)}'
    return f'{source_name}\n{verdict} (fit SNR {snr:.1f})'


def plot_faraday_depth_spectra(fit_params, catalog, test_rm_grid, faraday_depth, titles):
    '''One panel per fit source, titled with titles[source_name]: its Faraday dispersion function around the fit RM,
    with the catalog and fit RMs marked.'''
    for si, source_name in enumerate(fit_params):
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(test_rm_grid[si], faraday_depth[si])
        ax.axvline(catalog[source_name]["rotation_measure"], color='k', linestyle='--',
                   label=f'Catalog RM: {catalog[source_name]["rotation_measure"]:.2f}')
        ax.axvline(fit_params[source_name]["rotation_measure"], color='r', linestyle='--',
                   label=f'Fit RM: {fit_params[source_name]["rotation_measure"]:.2f}')
        ax.set_xlabel('Rotation Measure (rad/m^2)')
        ax.set_ylabel(f'Faraday Response of {FIT_POL_LABEL} ({hd_meta.vis_units})')
        ax.set_title(titles[source_name])
        ax.legend()
        plt.tight_layout()
        display(fig)
        plt.close(fig)


def plot_source_images(fit_params, catalog, images, titles):
    '''One panel per source, titled with titles[source_name]: its dirty image (2 degrees across, phased to the fit position
    and de-rotated at the fit RM), with the catalog and fit positions marked.'''
    for si, source_name in enumerate(fit_params):
        ra, dec = fit_params[source_name]["right_ascension"], fit_params[source_name]["declination"]
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(np.abs(images[si])[::-1], cmap='turbo', interpolation='none', aspect='auto', extent=[ra - 1, ra + 1, dec - 1, dec + 1])
        ax.set_title(titles[source_name])
        ax.set_xlabel('Right Ascension (deg)')
        ax.set_ylabel('Declination (deg)')
        ax.scatter(catalog[source_name]['right_ascension'], catalog[source_name]['declination'], color='white', marker='o',
                   label=f'Catalog position (RA, DEC): ({catalog[source_name]["right_ascension"]:.2f}, {catalog[source_name]["declination"]:.2f})')
        ax.scatter(ra, dec, color='white', marker='x', s=80, label=f'Fit position (RA, DEC): ({ra:.2f}, {dec:.2f})')
        ax.legend(fontsize=8)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

In [ ]:
# Build a monotonically increasing LST array, unwrapping across midnight if needed
lst_array = hd_meta.lsts.copy()
if lst_array[0] > lst_array[-1]:
    lst_midpoint = (lst_array[0] + lst_array[-1]) / 2
    lst_array[lst_array > lst_midpoint] -= 2 * np.pi

In [ ]:
best_fit_source_params = {}
source_images = []
accepted_fits = []  # sources whose fit passed every check
fit_titles = {}  # figure titles saying whether each source's fit was kept
snr_image_fov = 20  # in degrees, the width of the dirty image each fit's SNR is measured in

print(f'A source\'s fit is kept only if it passes every check below; otherwise its catalog values are used. Its SNR is '
      f'the amplitude at the fit position of a {snr_image_fov}-degree dirty image of {FIT_POL_LABEL} (pI for prior sources), '
      'phased to the fit position and de-rotated at the fit RM, over the MAD-based rms of that image.\n')

for source_name in data_list:
    lst_center = sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_unflagged_values(hd_meta.lsts, lst_center, combined_time_flags, k=FIT_WINDOW_TIMES)

    # Guard: skip iterative fit entirely if all data in this window is flagged
    prior_slice = prior_flags[imin:imax][:, FREQ_SLICE]
    if not has_usable_data(flags_list[source_name], nsamples_list[source_name], prior_slice):
        print(f"{source_name}: all data flagged in fit window — skipping fit, using catalog values.")
        fit_titles[source_name] = f'{source_name}\ncatalog values used (all data flagged)'
        best_fit_source_params[source_name] = {
            "right_ascension":         sources[source_name]["right_ascension"],
            "declination":             sources[source_name]["declination"],
            "rotation_measure":        sources[source_name]["rotation_measure"],
            "fit_scintillation":       sources[source_name]["fit_scintillation"],
            "scintillation_timescale": sources[source_name]["scintillation_timescale"],
            "lst_half_width":          sources[source_name]["lst_half_width"],
            "use_sidelobe_model":      sources[source_name]["use_sidelobe_model"],
        }
        continue

    fit_ra, fit_dec, fit_rm, converged = fit_polarized_source_params(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        sources[source_name]['right_ascension'],
        sources[source_name]['declination'],
        sources[source_name]['rotation_measure'],
        hd_meta.telescope.location,
        u_sign=U_SIGN,
        maxiter=FIT_MAXITER,
    )

    # Use fit parameters to check image to evaluate fit
    images = image_polarized_data(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        fit_ra, 
        fit_dec,
        fit_rm,
        hd_meta.telescope.location,
        u_sign=U_SIGN,
        fov=snr_image_fov,
    )
    avg_image = np.squeeze(images.images)
    sigma = np.sqrt((mad_std(avg_image.real) ** 2 + mad_std(avg_image.imag) ** 2) / 2)
    central_pixel = np.abs(avg_image[avg_image.shape[0] // 2, avg_image.shape[1] // 2])
    
    # Store the image
    source_images.append(avg_image)

    # Keep the fit only if it passes every check; otherwise fall back to the catalog values
    failed = report_fit_checks(source_name, [
        ('SNR', central_pixel / sigma, '.1f', '>', MIN_SNR, ''),
        ('RA - catalog', fit_ra - sources[source_name]["right_ascension"], '+.4f', 'within', MAX_RA_DIFF, 'deg'),
        ('Dec - catalog', fit_dec - sources[source_name]["declination"], '+.4f', 'within', MAX_DEC_DIFF, 'deg'),
        ('RM - catalog', fit_rm - sources[source_name]["rotation_measure"], '+.2f', 'within', MAX_RM_DIFF, 'rad/m^2'),
    ], f'fit RA {fit_ra:.4f} deg, Dec {fit_dec:.4f} deg, RM {fit_rm:.2f} rad/m^2'
       + ('' if converged else f'; not converged after {FIT_MAXITER} iterations'))
    fit_titles[source_name] = fit_title(source_name, central_pixel / sigma, failed)
    if not failed:
        accepted_fits.append(source_name)
        best_fit_source_params[source_name] = {
            "right_ascension":  fit_ra,
            "declination":      fit_dec,
            "rotation_measure": fit_rm,
            "fit_scintillation": sources[source_name]["fit_scintillation"],
            "scintillation_timescale": sources[source_name]["scintillation_timescale"],
            "lst_half_width": sources[source_name]["lst_half_width"],
            "use_sidelobe_model": sources[source_name]["use_sidelobe_model"]
        }
    else:
        best_fit_source_params[source_name] = {
            "right_ascension":  sources[source_name]["right_ascension"],
            "declination":      sources[source_name]["declination"],
            "rotation_measure": sources[source_name]["rotation_measure"],
            "fit_scintillation": sources[source_name]["fit_scintillation"],
            "scintillation_timescale": sources[source_name]["scintillation_timescale"],
            "lst_half_width": sources[source_name]["lst_half_width"],
            "use_sidelobe_model": sources[source_name]["use_sidelobe_model"]
        }

prior_best_fit_source_params = {}
prior_fit_titles = {}
for source_name in prior_sources:
    if not prior_sources[source_name]["fit_position"]:
        print(f"{source_name}: fewer than {MIN_TIMES} usable integrations in its own LST window — using catalog values.")
        prior_fit_titles[source_name] = f'{source_name}\ncatalog values used (fewer than {MIN_TIMES} usable integrations)'
        prior_best_fit_source_params[source_name] = {
            "right_ascension": prior_sources[source_name]["right_ascension"],
            "declination":     prior_sources[source_name]["declination"],
            "lst_half_width":  prior_sources[source_name]["lst_half_width"],
        }
        continue

    lst_center = prior_sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_unflagged_values(hd_meta.lsts, lst_center, combined_time_flags, k=FIT_WINDOW_TIMES)

    # Guard: skip iterative fit entirely if all data in this window is flagged
    prior_slice = prior_flags[imin:imax][:, FREQ_SLICE]
    if not has_usable_data(prior_flags_list[source_name], prior_nsamples_list[source_name], prior_slice):
        print(f"{source_name}: all data flagged in fit window — skipping fit, using catalog values.")
        prior_fit_titles[source_name] = f'{source_name}\ncatalog values used (all data flagged)'
        prior_best_fit_source_params[source_name] = {
            "right_ascension": prior_sources[source_name]["right_ascension"],
            "declination":     prior_sources[source_name]["declination"],
            "lst_half_width":  prior_sources[source_name]["lst_half_width"],
        }
        continue

    fit_ra, fit_dec, _ = polfilt.iteratively_fit_polarized_source_params(
        prior_data_list[source_name],
        prior_flags_list[source_name],
        prior_nsamples_list[source_name],
        prior_sources[source_name]['right_ascension'],
        prior_sources[source_name]['declination'],
        0.0, # RM component is zero for bright interloper sources
        hd_meta.telescope.location,
        maxiter=FIT_MAXITER,
        verbose=False,
        drm=1e-5, # something small to not diverge from 0.0
        dtest=10, # something small to reduce rm fit time
        pol='pI'
    )

    # Use fit parameters to check image to evaluate fit
    images = image_polarized_data(
        prior_data_list[source_name],
        prior_flags_list[source_name],
        prior_nsamples_list[source_name],
        fit_ra, 
        fit_dec,
        0.0, # RM component is zero for bright interloper sources
        hd_meta.telescope.location,
        pol="pI",
        fov=snr_image_fov,
    )
    avg_image = np.squeeze(images.images)
    sigma = np.sqrt((mad_std(avg_image.real) ** 2 + mad_std(avg_image.imag) ** 2) / 2)
    central_pixel = np.abs(avg_image[avg_image.shape[0] // 2, avg_image.shape[1] // 2])
    
    # Keep the fit only if it passes every check; otherwise fall back to the catalog values
    failed = report_fit_checks(source_name, [
        ('SNR', central_pixel / sigma, '.1f', '>', MIN_SNR, ''),
        ('RA - catalog', fit_ra - prior_sources[source_name]["right_ascension"], '+.4f', 'within', MAX_RA_DIFF, 'deg'),
        ('Dec - catalog', fit_dec - prior_sources[source_name]["declination"], '+.4f', 'within', MAX_DEC_DIFF, 'deg'),
    ], f'fit RA {fit_ra:.4f} deg, Dec {fit_dec:.4f} deg')
    prior_fit_titles[source_name] = fit_title(source_name, central_pixel / sigma, failed)
    if not failed:
        prior_best_fit_source_params[source_name] = {
            "right_ascension":  fit_ra,
            "declination":      fit_dec,
        }
    else:
        prior_best_fit_source_params[source_name] = {
            "right_ascension":  prior_sources[source_name]["right_ascension"],
            "declination":      prior_sources[source_name]["declination"],
            "lst_half_width": prior_sources[source_name]["lst_half_width"]
        }

In [ ]:
source_images = []
for source_name in best_fit_source_params:
    lst_center = sources[source_name]['right_ascension'] / 15 * np.pi / 12
    imin, imax = closest_values(hd_meta.lsts, lst_center, k=FIT_WINDOW_TIMES)
    
    images = image_polarized_data(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        best_fit_source_params[source_name]["right_ascension"], 
        best_fit_source_params[source_name]["declination"],
        best_fit_source_params[source_name]["rotation_measure"],
        hd_meta.telescope.location,
        u_sign=U_SIGN,
        fov=2,
        npix=100
    )
    source_images.append(np.squeeze(images.images))

In [ ]:
prior_source_images = []
for source_name in prior_data_list:  # the prior sources loaded to fit their positions
    images = image_polarized_data(
        prior_data_list[source_name],
        prior_flags_list[source_name],
        prior_nsamples_list[source_name],
        prior_best_fit_source_params[source_name]["right_ascension"], 
        prior_best_fit_source_params[source_name]["declination"],
        0.0,
        hd_meta.telescope.location,
        fov=2,
        npix=100,
        pol='pI'
    )
    prior_source_images.append(np.squeeze(images.images))

In [ ]:
faraday_depth = []
test_rm_grid  = []
rm_resolution = 2 * np.sqrt(3) / np.ptp((constants.c.value / hd_meta.freqs[FREQ_SLICE]) ** 2)  # FWHM of the RM spread function, in rad/m^2
mirrored = []

for source_name in best_fit_source_params:
    fit_rm = best_fit_source_params[source_name]["rotation_measure"]
    spectrum_freqs, spectrum = phased_spectrum(
        data_list[source_name],
        flags_list[source_name],
        nsamples_list[source_name],
        best_fit_source_params[source_name]["right_ascension"],
        best_fit_source_params[source_name]["declination"],
        hd_meta.telescope.location,
        u_sign=U_SIGN,
    )
    test_rm = np.linspace(fit_rm - 5, fit_rm + 5, 500)  # in rad/m^2
    test_rm_grid.append(test_rm)
    faraday_depth.append(faraday_response(spectrum, spectrum_freqs, test_rm))

    # pQ + i pU holds a source at its RM alone, so a detected source whose RM is resolved should respond far less at minus
    # its RM; responding more there means pU enters with the wrong sign, or the E-N cross-polarization phase is off
    if U_SIGN != 0 and source_name in accepted_fits and abs(fit_rm) > rm_resolution:
        at_fit, at_mirror = faraday_response(spectrum, spectrum_freqs, [fit_rm, -fit_rm])
        print(f'{source_name}: Faraday response of {FIT_POL_LABEL} at minus the fit RM ({-fit_rm:.2f} rad/m^2) is '
              f'{at_mirror / at_fit:.2f}x that at the fit RM.')
        if at_mirror > at_fit:
            mirrored.append(source_name)

if mirrored:
    warnings.warn(f'{", ".join(mirrored)}: Faraday response at minus the fit RM exceeds that at the fit RM; check '
                  f'PSEUDO_STOKES_U_SIGN = {PSEUDO_STOKES_U_SIGN} and the E-N cross-polarization phase calibration.')

# *Figure 1: Faraday Depth Spectra*

For each polarized source, the amplitude of its visibilities in the fit polarization (pQ + i pU, or pQ with
`FIT_PSEUDO_STOKES_U = False`), phased to the fit position, averaged over baselines and the fit window's
integrations (samples flagged in any polarization used or in the sky-filtered flags get zero weight), and
de-rotated by each trial RM within 5 rad/m^2 of the fit RM, over `FIT_FREQ_RANGE`: the quantity the RM fit
maximizes. Dashed lines mark the catalog (black) and fit (red) RMs; a detected source peaks at the fit RM. Each
title says whether the fit passed the checks printed above and gives its SNR; for a rejected fit, the red line is
the catalog RM the rest of the notebook uses.

In [ ]:
plot_faraday_depth_spectra(best_fit_source_params, sources, test_rm_grid, faraday_depth, fit_titles)

# *Figure 2: Source Images and Fit Positions*

The amplitude of each polarized source's dirty image, 2 degrees across, made from the visibilities of Figure 1
phased to the fit position and de-rotated at the fit RM; in pQ + i pU this is polarized intensity. The circle
marks the catalog position and the cross the fit position, which should sit on the peak. Titles as in Figure 1;
for a rejected fit, the image is centred on the catalog position and the cross marks it.

In [ ]:
plot_source_images(best_fit_source_params, sources, source_images, fit_titles)

# *Figure 3: Prior Source Images and Fit Positions*

In [ ]:
# the prior sources at their catalog positions have no data around their transits to image
plot_source_images({name: prior_best_fit_source_params[name] for name in prior_data_list}, prior_sources, prior_source_images, prior_fit_titles)

## Model Sources w/ Sidelobes Intersecting Polarized Sources

1. User must provide a list of sources to be modeled and subtracted from the visibilities (this notebook does not identify the sources)
2. For each source in the list, phase to the source and low-pass delay filter the phased waterfall to remove residual sidelobes of other sources
3. Subtract the model of the source from the visibilities, then move on to the next source

In [ ]:
import warnings
import functools
from multiprocessing import Pool
from copy import deepcopy
from hera_cal import datacontainer


def _process_file_hera_with_phasor(
    filepath, target_source, subtraction_sources=None, lst_range=None
):
    """Phase-rotate and accumulate weighted visibilities toward a single target source,
    optionally after subtracting foreground model visibilities.

    Reads the LST window specified by *lst_range* (or the full file if ``None``),
    optionally subtracts a set of modelled point sources from the raw visibilities,
    then computes the geometric phase for the baseline toward the target source at
    each time/frequency sample and returns weighted numerators and denominators
    ready for averaging across baselines.

    Parameters
    ----------
    filepath : str
        Path to a single-baseline HERA UVH5 data file.
    target_source : dict
        The single source to phase toward.  Expected keys:

        ``right_ascension`` : float
            J2000 right ascension in degrees.
        ``declination`` : float
            J2000 declination in degrees.

    subtraction_sources : dict or None
        Point sources to subtract before phasing.  Keys are source names;
        values are dicts with keys:

        ``right_ascension`` : float – J2000 RA in degrees.
        ``declination``     : float – J2000 Dec in degrees.
        ``ee``, ``nn``      : array_like, shape (n_model_times, n_freqs)
            Phased, baseline-averaged model visibilities.
        ``times``           : array_like, shape (n_model_times,)
            JDs of the model rows; each is subtracted at the loaded
            integration with the same JD (``np.intersect1d``).

        Pass ``None`` (default) to skip subtraction.

    lst_range : tuple of float or None
        ``(lst_min, lst_max)`` in radians.  Only integrations whose LST falls
        within this window are loaded, which can substantially reduce I/O when
        you only need the times overlapping a set of polarized sources.
        Pass ``None`` to read the full file.

    Returns
    -------
    out_num, out_denom, times : tuple
        Dicts of ``{pol: array}`` for the weighted visibility numerator and
        the weight denominator, and the time axis of the loaded window.
    """
    hd = io.HERAData(filepath)

    # Only load the requested LST window (default: whole file).
    if lst_range is not None:
        read_kwargs = dict(polarizations=["ee", "nn"], lst_range=lst_range)
    else:
        read_kwargs = dict(polarizations=["ee", "nn"])

    d, f, n = hd.read(**read_kwargs)
    time_inds = np.searchsorted(hd.times, d.times)  # the night's flag waterfall, at the loaded times

    ap     = list(d.antpairs())[0]   # single baseline per file
    bl_vec = hd.antpos[ap[1]] - hd.antpos[ap[0]]

    # ------------------------------------------------------------------ #
    # Subtract all foreground model visibilities before phasing.          #
    # ------------------------------------------------------------------ #
    if subtraction_sources is not None:
        for sub_name, sub_info in subtraction_sources.items():
            _, data_ind, model_ind = np.intersect1d(
                d.times,
                sub_info["times"],
                return_indices=True
            )

            l_s, m_s, n_s = polfilt.radec_to_lmn(
                sub_info["right_ascension"],
                sub_info["declination"],
                sub_info["times"][model_ind],
                hd.telescope.location,
            )
            lmn_s = np.stack([l_s, m_s, n_s])            # (3, n_times)

            # Positive sign: forward-model phasor. (n_times, n_freqs)
            phase_s = (bl_vec @ lmn_s / constants.c.value * d.freqs[:, None]).T

            for pol in ["ee", "nn"]:
                model = sub_info[pol][model_ind]
                #time_inds = np.searchsorted(hd.times, sub_info["times"])
                key = ap + (pol,)
                d[key][data_ind] -= model * np.exp(2j * np.pi * phase_s)

    # ------------------------------------------------------------------ #
    # Phase-rotate toward the single target source.                       #
    # ------------------------------------------------------------------ #
    l0, m0, n0 = polfilt.radec_to_lmn(
        target_source["right_ascension"],
        target_source["declination"],
        d.times,
        hd.telescope.location,
    )
    lmn0   = np.stack([l0, m0, n0])                      # (3, n_times)
    phase  = (bl_vec @ lmn0 / constants.c.value * d.freqs[:, None]).T
    phasor = np.exp(-2j * np.pi * phase)

    out_num   = {}
    out_denom = {}

    for pol in ["ee", "nn"]:
        key     = ap + (pol,)
        vis_bl  = d[key]
        flag_bl = f[key] | prior_flags[time_inds]
        nsamp   = n[key]

        valid  = (
            ~flag_bl
            & np.isfinite(vis_bl)
            & np.isfinite(nsamp)
            & (nsamp > 0)
        )
        w      = np.where(valid, nsamp, 0.0)
        vis_bl = np.where(np.isfinite(vis_bl), vis_bl, 0.0)

        out_num[pol]   = w * (vis_bl * phasor + np.conj(vis_bl * phasor)) / 2
        out_denom[pol] = w

    return out_num, out_denom, d.times


def load_parallel_average_over_baseline(
    files, target_source, max_workers=8, subtraction_sources=None, lst_range=None
):
    """Load files in parallel and form a baseline-averaged phase-rotated dataset,
    optionally with foreground subtraction applied before phasing.

    Accumulates weighted visibility numerators and denominators across all
    baselines, then divides to obtain the average phase-rotated spectrum for
    the target source.

    Parameters
    ----------
    files : list of str
        Single-baseline UVH5 files to load.
    target_source : dict
        The single source to phase toward; see
        :func:`_process_file_hera_with_phasor` for the expected format.
    max_workers : int
        Number of parallel worker processes.
    subtraction_sources : dict or None
        Point sources to subtract from each baseline before phasing; see
        :func:`_process_file_hera_with_phasor` for the expected format.
        Passing ``None`` (default) disables subtraction.
    lst_range : tuple of float or None
        ``(lst_min, lst_max)`` in radians passed through to each worker so
        that only integrations within the window are read.  ``None`` loads
        the full file.

    Returns
    -------
    data_list, flags_list, nsamples_list, times : tuple
        Averaged visibilities, combined flags, accumulated sample weights
        (all as ``{pol: array}`` dicts), and the time array.
    """
    num   = None
    denom = None
    times = None

    # Bind extra arguments so the worker signature stays (filepath,).
    worker = functools.partial(
        _process_file_hera_with_phasor,
        target_source=target_source,
        subtraction_sources=subtraction_sources,
        lst_range=lst_range,
    )

    with Pool(processes=max_workers) as pool:
        results = pool.imap_unordered(worker, files, chunksize=1)
        for result in tqdm(results, total=len(files)):
            if result is None:
                continue
            try:
                out_num, out_denom, out_times = result
                if num is None:
                    num, denom, times = out_num, out_denom, out_times
                else:
                    for pol in ["ee", "nn"]:
                        num[pol]   += out_num[pol]
                        denom[pol] += out_denom[pol]
            except Exception as exc:
                print(f"Merge error: {exc}")
                raise

    data_list = {}
    flags_list = {}
    nsamples_list = {}
    for pol in ["ee", "nn"]:
        valid_denom = np.isfinite(denom[pol]) & (denom[pol] > 0)
        clean_denom = np.where(valid_denom, denom[pol], 0.0)
        average = np.zeros_like(num[pol], dtype=complex)
        np.divide(num[pol], clean_denom, out=average, where=valid_denom)

        # A non-finite numerator invalidates the corresponding average even
        # when its accumulated denominator happens to be positive.
        valid_average = valid_denom & np.isfinite(average)
        data_list[pol] = np.where(valid_average, average, 0.0)
        flags_list[pol] = ~valid_average
        nsamples_list[pol] = np.where(valid_average, clean_denom, 0.0)

    return data_list, flags_list, nsamples_list, times

In [ ]:
# Compute the union LST window that covers every polarized source that uses the sidelobe model.
# Loading only these integrations when building the sidelobe models
# avoids reading times that are irrelevant to the polarized-source fits.
pol_lst_centers = [
    best_fit_source_params[s]["right_ascension"] / 15 * np.pi / 12
    for s in best_fit_source_params 
    if best_fit_source_params[s]["use_sidelobe_model"]
]
pol_lst_half_widths = [
    best_fit_source_params[s]["lst_half_width"]
    for s in best_fit_source_params
    if best_fit_source_params[s]["use_sidelobe_model"]
]

if len(pol_lst_centers) > 0:
    pol_lst_min = min(c - hw for c, hw in zip(pol_lst_centers, pol_lst_half_widths))
    pol_lst_max = max(c + hw for c, hw in zip(pol_lst_centers, pol_lst_half_widths))
    pol_lst_range = (pol_lst_min, pol_lst_max)
    print(f"Union LST range for polarized sources: "
          f"{pol_lst_range[0] * 12 / np.pi:.4f}h – {pol_lst_range[1] * 12 / np.pi:.4f}h")
     
    subtraction_sources = {}
     
    for prior_source in prior_best_fit_source_params:
        data_list, flag_list, nsamples_list, times = load_parallel_average_over_baseline(
            load_files,
            target_source=prior_best_fit_source_params[prior_source],
            max_workers=MAX_WORKERS,
            subtraction_sources=subtraction_sources,
            lst_range=pol_lst_range,
        )
     
        subtraction_sources[prior_source] = {}
        data_list_filtered = deepcopy(data_list)
     
        for pol in data_list_filtered:
            # If this polarization is fully flagged over the loaded LST window,
            # there is no information to fit. Store an explicit zero model and
            # skip the DPSS filter instead of passing None bands to fourier_filter.
            data_list_filtered[pol][flag_list[pol]] = 0.0
            if np.all(flag_list[pol]):
                print(f"  {prior_source} {pol}: all data flagged in sidelobe window; using zero model.")
                subtraction_sources[prior_source][pol] = np.zeros_like(data_list_filtered[pol])
                continue

            # Split into low/high sub-bands around the FM gap
            _, FILTER_BANDS = flag_utils.get_minimal_slices(
                flag_list[pol], freqs=hd_meta.freqs,
                freq_cuts=[BAND_SPLIT_FREQ * 1e6],
            )
            for band in FILTER_BANDS:
                if band is None:
                    continue
                # flagged cells get zero weight (no inpainted values to lean on)
                data_list_filtered[pol][:, band], _, _ = dspec.fourier_filter(
                    hd_meta.freqs[band],
                    data_list[pol][:, band],
                    np.where(flag_list[pol][:, band], 0.0, nsamples_list[pol][:, band]),
                    [0],
                    [SIDELOBE_DELAY_HALF_WIDTH * 1e-9],
                    eigenval_cutoff=[EIGENVAL_CUTOFF],
                    mode="dpss_matrix",
                    ridge_alpha=EIGENVAL_CUTOFF,
                    max_contiguous_edge_flags=hd_meta.freqs[band].size,
                )

            data_list_filtered[pol][flag_list[pol]] = 0.0
            subtraction_sources[prior_source][pol] = np.copy(data_list_filtered[pol])
     
        subtraction_sources[prior_source]["right_ascension"] = prior_best_fit_source_params[prior_source]["right_ascension"]
        subtraction_sources[prior_source]["declination"] = prior_best_fit_source_params[prior_source]["declination"]
        subtraction_sources[prior_source]["times"] = hd_meta.times[(hd_meta.lsts > pol_lst_range[0]) & (hd_meta.lsts < pol_lst_range[1])]
else:
    subtraction_sources = {}

## Build Faraday-Rotating DPSS Source Models

With best-fit positions and rotation measures in hand, we now:

* load a wider LST window of data, each source's catalog `lst_half_width` (0.75 hours for the current catalog) on either side of its transit;
* average over all baselines, phase-rotating each to the source;
* fit a separable DPSS model in time x frequency that captures the Faraday-modulated source signal.

In [ ]:
# Note: _process_file_hera_with_phasor and load_parallel_average_over_baseline
# are defined in the 'Model Sources w/ Sidelobes' section above and reused here

def _dpss_fit(data_2d, weights, time_basis, freq_basis, solver="pinv", return_beta=False):
    """Fit a separable DPSS model to a 2-D (time x frequency) visibility array.

    Solves the weighted least-squares problem

        min_beta  || W^(1/2) (y - (T x F) beta) ||^2

    via a pseudo-inverse, where *T* and *F* are the time and frequency DPSS
    basis matrices respectively.

    Parameters
    ----------
    data_2d : ndarray, shape (n_times, n_freqs)
        Complex visibility array.
    weights : ndarray, shape (n_times, n_freqs)
        Real weight array.
    time_basis : ndarray, shape (n_times, n_time_comps)
        DPSS basis matrix in the time direction.
    freq_basis : ndarray, shape (n_freqs, n_freq_comps)
        DPSS basis matrix in the frequency direction.
    solver : str
        Linear solver passed to ``_linear_fit`` (default: ``"pinv"``)

    Returns
    -------
    ndarray, shape (n_times, n_freqs)
        Model visibilities.
    """
    valid = (
        np.isfinite(data_2d)
        & np.isfinite(weights)
        & (weights > 0)
    )
    data_2d = np.where(valid, data_2d, 0.0)
    weights = np.where(valid, weights, 0.0)
    if not np.any(weights):
        model = np.zeros_like(data_2d, dtype=complex)
        beta = np.zeros(
            (time_basis.shape[-1], freq_basis.shape[-1]), dtype=complex
        )
        return (model, beta) if return_beta else model
    weights = weights / np.max(weights)

    XTX = np.einsum(
        "ti,fj,tf,tm,fn->ijmn",
        time_basis.conj(), freq_basis.conj(),
        weights,
        time_basis,       freq_basis,
        optimize=True,
    )
    n_comps = time_basis.shape[-1] * freq_basis.shape[-1]
    XTX     = XTX.reshape(n_comps, n_comps)
    XTX     = 0.5 * (XTX + XTX.conj().T)

    # X^T W y via the Kronecker identity: (A x B) vec(Y) = A Y B^T
    XTWy = np.ravel(time_basis.conj().T @ (data_2d * weights) @ freq_basis.conj())

    if solver == "pinv":
        beta = np.linalg.pinv(XTX, hermitian=True) @ XTWy
    else:
        beta, _ = _linear_fit(XTX, XTWy, solver=solver)
    beta    = beta.reshape(time_basis.shape[-1], freq_basis.shape[-1])

    if return_beta:
        return time_basis @ beta @ freq_basis.T, beta
    else:
        return time_basis @ beta @ freq_basis.T


def _time_dependent_freq_fit(
    data_2d, weights, time_basis, freq_basis_3d, solver="pinv",
    return_beta=False,
):
    """Fit a separable time model with a frequency basis that varies by time.

    ``freq_basis_3d[t]`` is the physical RM basis after applying integration
    ``t``'s weighted foreground projection.  Allowing it to vary is necessary
    when flags or relative filter weights change with time.
    """
    basis_valid = np.all(np.isfinite(freq_basis_3d), axis=-1)
    valid = (
        np.isfinite(data_2d)
        & np.isfinite(weights)
        & (weights > 0)
        & basis_valid
    )
    data_2d = np.where(valid, data_2d, 0.0)
    weights = np.where(valid, weights, 0.0)
    freq_basis_3d = np.where(np.isfinite(freq_basis_3d), freq_basis_3d, 0.0)
    if not np.any(weights):
        model = np.zeros_like(data_2d, dtype=complex)
        beta = np.zeros(
            (time_basis.shape[-1], freq_basis_3d.shape[-1]), dtype=complex
        )
        return (model, beta) if return_beta else model
    weights = weights / np.max(weights)

    XTX = np.einsum(
        "ti,tfj,tf,tm,tfn->ijmn",
        time_basis.conj(), freq_basis_3d.conj(), weights,
        time_basis,        freq_basis_3d,
        optimize=True,
    )
    n_time_comps = time_basis.shape[-1]
    n_freq_comps = freq_basis_3d.shape[-1]
    XTX = XTX.reshape(n_time_comps * n_freq_comps, -1)
    XTX = 0.5 * (XTX + XTX.conj().T)

    XTWy = np.einsum(
        "ti,tfj,tf,tf->ij",
        time_basis.conj(), freq_basis_3d.conj(), weights, data_2d,
        optimize=True,
    ).ravel()

    if solver == "pinv":
        beta = np.linalg.pinv(XTX, hermitian=True) @ XTWy
    else:
        beta, _ = _linear_fit(XTX, XTWy, solver=solver)
    beta = beta.reshape(n_time_comps, n_freq_comps)
    model = np.einsum(
        "ti,ij,tfj->tf", time_basis, beta, freq_basis_3d, optimize=True
    )
    return (model, beta) if return_beta else model


def _foreground_ranked_rm_basis(
    freq_basis, rm_phasor, freqs, foreground_half_width, eigenval_cutoff,
    max_foreground_fraction=1e-3, rcond=1e-12,
):
    """Construct physical RM modes ranked by unweighted foreground overlap.

    The returned modes remain entirely inside the original two-branch RM
    subspace.  A mode is retained only when its squared unweighted projection
    into ``span(A_F)`` is at most ``max_foreground_fraction``.
    """
    A_F = dspec.dpss_operator(
        freqs, [0], [foreground_half_width],
        eigenval_cutoff=[eigenval_cutoff],
    )[0]
    Phi = np.concatenate(
        [rm_phasor[:, None] * freq_basis, rm_phasor.conj()[:, None] * freq_basis],
        axis=1,
    )

    # Work only with the two column spaces, so the result is invariant to
    # scaling or correlations among the original DPSS/RM columns.
    U_F, s_F, _ = np.linalg.svd(A_F, full_matrices=False)
    U_F = U_F[:, s_F > rcond * s_F[0]]
    U_source, s_source, _ = np.linalg.svd(Phi, full_matrices=False)
    U_source = U_source[:, s_source > rcond * s_source[0]]

    source_after_fg = U_source - U_F @ (U_F.conj().T @ U_source)
    _, rho, Vh = np.linalg.svd(source_after_fg, full_matrices=False)
    survival = np.clip(rho**2, 0.0, 1.0)
    foreground_fraction = 1.0 - survival
    keep = foreground_fraction <= max_foreground_fraction
    physical_basis = U_source @ Vh[keep].conj().T
    info = {
        "survival": survival,
        "foreground_fraction": foreground_fraction,
        "keep": keep,
        "n_input_modes": Phi.shape[1],
        "n_source_modes": U_source.shape[1],
    }
    return A_F, physical_basis, info


def _weighted_foreground_residual(
    X, A_F, weights, ridge_alpha=0.0, rcond=1e-15,
):
    """Apply the same weighted DPSS foreground removal used on the data.

    ``ridge_alpha`` follows ``hera_filters``' ``dpss_matrix`` convention:
    the diagonal of the weighted normal matrix is multiplied by
    ``1 + ridge_alpha``.
    """
    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(A_F)):
        raise ValueError("Foreground and source bases must be finite.")

    weights = np.where(
        np.isfinite(weights) & (weights > 0), weights, 0.0
    )
    weight_scale = np.max(weights, initial=0.0)
    if weight_scale <= 0:
        # With no constrained channels the foreground fit is identically
        # zero, so its residual operator is the identity.  This row will
        # also have zero fit weight and cannot affect the RM coefficients.
        return X.copy()
    weights = weights / weight_scale

    XTX = A_F.conj().T @ (weights[:, None] * A_F)
    # XTX is Hermitian by construction.  Enforce that symmetry before the
    # eigensolver so zero-heavy rows do not enter the less-stable general SVD.
    XTX = 0.5 * (XTX + XTX.conj().T)
    if ridge_alpha > 0:
        XTX = XTX.copy()
        XTX.flat[::XTX.shape[0] + 1] *= 1.0 + ridge_alpha
    XTy = A_F.conj().T @ (weights[:, None] * X)
    try:
        coefficients = (
            np.linalg.pinv(XTX, rcond=rcond, hermitian=True) @ XTy
        )
    except np.linalg.LinAlgError:
        # Fall back to the unsquared weighted design.  Its condition number
        # is better than that of XTX, and the augmented rows reproduce the
        # diagonal ridge convention above.
        sw = np.sqrt(weights)
        weighted_A = sw[:, None] * A_F
        weighted_X = sw[:, None] * X
        if ridge_alpha > 0:
            ridge_diag = np.sqrt(
                ridge_alpha
                * np.maximum(np.sum(np.abs(weighted_A) ** 2, axis=0), 0.0)
            )
            weighted_A = np.vstack([weighted_A, np.diag(ridge_diag)])
            weighted_X = np.vstack([
                weighted_X,
                np.zeros((A_F.shape[1], X.shape[1]), dtype=X.dtype),
            ])
        coefficients = np.linalg.lstsq(
            weighted_A, weighted_X, rcond=rcond
        )[0]
    return X - A_F @ coefficients


def _residualized_rm_model(
    data_2d, fit_weights, filter_weights, time_basis, A_F, physical_basis,
    ridge_alpha=0.0, solver="pinv", physical=True,
):
    """Fit filtered data with per-integration weighted RM basis responses.

    The coefficients are fit to each integration's weighted foreground-filter
    response of the physical RM modes, which is what the filtered data contain.
    Returns the source model and its part inside the foreground span.  With
    ``physical``, the model is the fit evaluated on the physical RM modes, i.e.
    the source itself, and its part inside the foreground span is what the
    foreground filter removed from it.  Otherwise the model is the fit evaluated
    on the residualized (filtered) modes, in foreground-filtered data space, and
    that part is zero.
    """
    valid = (
        np.isfinite(data_2d)
        & np.isfinite(fit_weights)
        & (fit_weights > 0)
    )
    if not np.any(valid):
        zero = np.zeros_like(data_2d, dtype=complex)
        return zero, zero.copy()
    data_2d = np.where(valid, data_2d, 0.0)
    fit_weights = np.where(valid, fit_weights, 0.0)

    filtered_basis = np.stack([
        _weighted_foreground_residual(
            physical_basis, A_F, filter_weights[t],
            ridge_alpha=ridge_alpha,
        )
        for t in range(data_2d.shape[0])
    ])

    filtered_model, beta = _time_dependent_freq_fit(
        data_2d, fit_weights, time_basis, filtered_basis,
        solver=solver, return_beta=True,
    )
    if not physical:
        return filtered_model, np.zeros_like(filtered_model)
    model = time_basis @ beta @ physical_basis.T
    return model, model - filtered_model


def trim_flagged_edges(flags_2d):
    """Return (t_start, t_end) with contiguously all-flagged edges removed.

    A leading/trailing integration is trimmed when every frequency channel in
    that row is flagged.  These rows add nothing to the DPSS fit but extend
    the time axis, reducing the effective number of well-conditioned
    eigenvectors.

    Parameters
    ----------
    flags_2d : ndarray, shape (n_times, n_freqs)
        Combined boolean flag array (True = flagged).

    Returns
    -------
    t_start, t_end : int
        Indices such that ``flags_2d[t_start:t_end]`` has no leading or
        trailing rows that are entirely True.
        Returns ``(0, 0)`` if every integration is fully flagged.
    """
    all_flagged = np.all(flags_2d, axis=1)  # (n_times,)

    if all_flagged.all():
        return 0, 0

    t_start = int(np.argmax(~all_flagged))
    t_end   = len(all_flagged) - int(np.argmax(~all_flagged[::-1]))
    return t_start, t_end


def _zero_model_result(hd):
    """Return a correctly shaped, all-zero model result for one source."""
    n_times = hd.times.shape[0]
    n_freqs = hd.freqs.shape[0]

    def zero_complex():
        return np.zeros((n_times, n_freqs), dtype=complex)

    def zero_weight():
        return np.zeros((n_times, n_freqs), dtype=float)

    return {
        "ee": {
            "fg_model":    zero_complex(),
            "rm_model":    zero_complex(),
            "rm_model_filtered": zero_complex(),
            "fg_filtered": zero_complex(),
            "scint_model": zero_complex(),
            "residual":    zero_complex(),
            "wgts":        zero_weight(),
        },
        "nn": {
            "fg_model":    zero_complex(),
            "rm_model":    zero_complex(),
            "rm_model_filtered": zero_complex(),
            "fg_filtered": zero_complex(),
            "scint_model": zero_complex(),
            "residual":    zero_complex(),
            "wgts":        zero_weight(),
        },
    }


def fit_pulsar_models(
    hd,
    avg_data_list,
    avg_flags_list,
    avg_nsamples_list,
    avg_times,
    temporal_half_width=0.5,
    spectral_half_width=50e-9,
    eigenval_cutoff=1e-12,
    foreground_half_width=750e-9,
):
    """Fit a Faraday-rotating DPSS model to each source in ``best_fit_source_params``.

    Frequency bands (FILTER_BANDS) are computed from the full-mask flag array
    before any time trimming, so the band layout is stable. Contiguously
    all-flagged integrations at the leading and trailing edges of the time window
    are trimmed before the DPSS fit to avoid spreading the eigenvectors over a
    range that contains no data; results are inserted back into full-night arrays.
    The data are not inpainted, so flagged cells are given zero weight in the
    foreground filter as well as in
    the source fit. Physical RM modes are selected using the unweighted
    source/foreground geometry, with an explicit maximum foreground-power
    fraction, and the retained basis is passed through each integration's weighted
    foreground filter before fitting, so its response matches the already-filtered
    data. With ``PHYSICAL_SOURCE_MODEL``, ``rm_model`` is the fit evaluated on the
    physical RM modes, and its part inside the foreground span is moved out of
    ``fg_model`` on unflagged cells, so ``fg_model + rm_model`` is unchanged there;
    otherwise ``rm_model`` is the fit evaluated on the residualized modes. Either way
    ``fg_filtered`` is the filtered data, and the scintillation model, the residual and
    ``rm_model_filtered`` (the Faraday model's filtered response) are in filtered space,
    so the scintillation model's part inside the foreground span stays in the data.
    Integrations the foreground filter skips (too few unflagged channels) still hold
    their foregrounds, so they get no weight in either fit and their basis is not
    residualized.

    Parameters
    ----------
    hd : HERAData
        Open data object used for frequency and LST metadata.
    avg_data_list, avg_flags_list, avg_nsamples_list : dict
        Per-source, per-polarisation averaged visibilities, flags, and weights.
    avg_times : dict
        Per-source arrays of UTC times (JD) for the time axis.
    temporal_half_width : float
        DPSS half-bandwidth in the time direction (mHz; default 0.5).
    spectral_half_width : float
        DPSS half-bandwidth in the frequency direction (seconds; default 50 ns).
    eigenval_cutoff : float
        Eigenvalue threshold for DPSS mode truncation.
    foreground_half_width : float
        Delay half-width (seconds) of the foreground DPSS filter applied before
        modelling.

    Returns
    -------
    dict
        Nested dict ``{source_name: {pol: {"rm_model",
        "fg_filtered", "wgts", ...}}}``, where
        each value is a full-night array of shape ``(n_times, n_freqs)``.
    """
    results = {}

    for source_name in avg_data_list:
        source_info = best_fit_source_params[source_name]

        # Build the LST mask for this source window
        lst_center = source_info["right_ascension"] / 15 * np.pi / 12
        lst_range  = (
            lst_center - source_info["lst_half_width"],
            lst_center + source_info["lst_half_width"],
        )
        mask         = (lst_range[0] < hd.lsts) & (hd.lsts < lst_range[1])
        n_mask_times = int(mask.sum())

        # Treat non-finite data and non-positive/non-finite sample counts as
        # flags before constructing bands or trimming time edges.
        data_full_ee = avg_data_list[source_name]["ee"]
        data_full_nn = avg_data_list[source_name]["nn"]
        nsamp_full_ee = avg_nsamples_list[source_name]["ee"]
        nsamp_full_nn = avg_nsamples_list[source_name]["nn"]
        invalid_ee = (
            ~np.isfinite(data_full_ee)
            | ~np.isfinite(nsamp_full_ee)
            | (nsamp_full_ee <= 0)
        )
        invalid_nn = (
            ~np.isfinite(data_full_nn)
            | ~np.isfinite(nsamp_full_nn)
            | (nsamp_full_nn <= 0)
        )
        flags_full_ee = (
            avg_flags_list[source_name]["ee"]
            | prior_flags[mask, :]
            | invalid_ee
        )
        flags_full_nn = (
            avg_flags_list[source_name]["nn"]
            | prior_flags[mask, :]
            | invalid_nn
        )
        # Use the common valid support for the two-polarization source fit.
        flags_full = flags_full_ee | flags_full_nn

        _, FILTER_BANDS = flag_utils.get_minimal_slices(
            flags_full,
            freqs=hd.freqs,
            freq_cuts=[BAND_SPLIT_FREQ * 1e6],
        )

        if all(band is None for band in FILTER_BANDS):
            print(f"  {source_name}: all frequency bands flagged in DPSS fit window; using zero model.")
            results[source_name] = _zero_model_result(hd)
            continue

        # Trim contiguously all-flagged edge integrations before fitting
        t_start, t_end = trim_flagged_edges(flags_full)
        n_fit_times    = t_end - t_start

        if n_fit_times < 2:
            print(
                f"  WARNING: {source_name} - only {n_fit_times} fit time(s) remain after "
                "edge-flag trimming; using zero model."
            )
            results[source_name] = _zero_model_result(hd)
            continue

        # Trimmed per-time arrays
        data_ee = np.where(
            np.isfinite(data_full_ee[t_start:t_end]),
            data_full_ee[t_start:t_end], 0.0,
        )
        data_nn = np.where(
            np.isfinite(data_full_nn[t_start:t_end]),
            data_full_nn[t_start:t_end], 0.0,
        )
        flags_here_ee = flags_full_ee[t_start:t_end]
        flags_here_nn = flags_full_nn[t_start:t_end]

        nsamp_ee_trim = np.where(
            np.isfinite(nsamp_full_ee[t_start:t_end])
            & (nsamp_full_ee[t_start:t_end] > 0),
            nsamp_full_ee[t_start:t_end], 0.0,
        )
        nsamp_nn_trim = np.where(
            np.isfinite(nsamp_full_nn[t_start:t_end])
            & (nsamp_full_nn[t_start:t_end] > 0),
            nsamp_full_nn[t_start:t_end], 0.0,
        )
        n_ee = np.where(flags_here_ee, 0.0, nsamp_ee_trim)
        n_nn = np.where(flags_here_nn, 0.0, nsamp_nn_trim)

        # The data are not inpainted, so flagged cells carry no information: the foreground filter weights them
        # to zero, the same as the fit weights.
        filt_wgts_ee = n_ee
        filt_wgts_nn = n_nn

        times_msec = avg_times[source_name][t_start:t_end] * 24 * 3600 / 1e3

        # Per-band foreground filter on the trimmed data, noting the integrations it skips
        fg_model_ee = np.zeros_like(data_ee)
        fg_model_nn = np.zeros_like(data_nn)
        unfiltered_ee = np.zeros_like(flags_here_ee)
        unfiltered_nn = np.zeros_like(flags_here_nn)

        for band in FILTER_BANDS:
            if band is None:
                continue
            filter_kwargs = dict(
                filter_centers=[0],
                filter_half_widths=[foreground_half_width],
                eigenval_cutoff=[eigenval_cutoff],
                mode="dpss_matrix",
                ridge_alpha=eigenval_cutoff,
                max_contiguous_edge_flags=hd.freqs[band].size,
            )
            for data_pol, filt_wgts_pol, fg_model_pol, unfiltered_pol in [
                (data_ee, filt_wgts_ee, fg_model_ee, unfiltered_ee),
                (data_nn, filt_wgts_nn, fg_model_nn, unfiltered_nn),
            ]:
                usable_rows = np.any(filt_wgts_pol[:, band] > 0, axis=1)
                if not np.any(usable_rows):
                    continue
                filtered_rows, _, filter_info = dspec.fourier_filter(
                    hd.freqs[band],
                    data_pol[usable_rows, band],
                    filt_wgts_pol[usable_rows, band],
                    **filter_kwargs,
                )
                fg_model_pol[usable_rows, band] = filtered_rows
                # fourier_filter leaves integrations with too few unflagged channels (skip_wgt) unfiltered
                skipped = [row for row, status in filter_info['status']['axis_1'].items() if status == 'skipped']
                unfiltered_pol[np.flatnonzero(usable_rows)[skipped], band] = True

        # Intermediate output arrays (n_mask_times x n_freqs).
        # The trimmed-off edge integrations stay at zero — correct since they
        # are entirely flagged.
        n_freqs = hd.freqs.shape[0]

        rm_model_ee_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)
        rm_model_nn_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)
        rm_filt_ee_out     = np.zeros((n_mask_times, n_freqs), dtype=complex)
        rm_filt_nn_out     = np.zeros((n_mask_times, n_freqs), dtype=complex)
        scint_model_ee_out = np.zeros((n_mask_times, n_freqs), dtype=complex)
        scint_model_nn_out = np.zeros((n_mask_times, n_freqs), dtype=complex)
        fg_filt_ee_out     = np.zeros((n_mask_times, n_freqs), dtype=complex)
        fg_filt_nn_out     = np.zeros((n_mask_times, n_freqs), dtype=complex)
        wgts_ee_out        = np.zeros((n_mask_times, n_freqs), dtype=float)
        wgts_nn_out        = np.zeros((n_mask_times, n_freqs), dtype=float)
        residual_ee_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)
        residual_nn_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)
        fg_model_ee_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)
        fg_model_nn_out    = np.zeros((n_mask_times, n_freqs), dtype=complex)

        fg_model_ee_out[t_start:t_end] = fg_model_ee
        fg_model_nn_out[t_start:t_end] = fg_model_nn

        data_ee_filt = np.where(flags_here_ee, 0.0 + 0.0j, data_ee - fg_model_ee)
        data_nn_filt = np.where(flags_here_nn, 0.0 + 0.0j, data_nn - fg_model_nn)

        if source_info["fit_scintillation"]:
            scintillation_hw   = 1e3 / source_info["scintillation_timescale"]
            time_filter_kwargs = dict(
                filter_centers=[0],
                filter_half_widths=[scintillation_hw],
                eigenval_cutoff=[eigenval_cutoff],
                mode="dpss_matrix",
                ridge_alpha=eigenval_cutoff,
                max_contiguous_edge_flags=times_msec.size,
                filter_dims=0,
            )
        else:
            time_filter_kwargs = {}

        for band in FILTER_BANDS:
            if band is None:
                continue
            time_basis = dspec.dpss_operator(
                times_msec, [0], [temporal_half_width], eigenval_cutoff=[eigenval_cutoff]
            )[0]
            freq_basis = dspec.dpss_operator(
                hd.freqs[band], [0], [spectral_half_width], eigenval_cutoff=[eigenval_cutoff]
            )[0]

            rm_phasor = np.exp(
                2j * (constants.c.value / hd.freqs[band]) ** 2
                * source_info["rotation_measure"]
            )

            fg_filt_ee = data_ee_filt[:, band]
            fg_filt_nn = data_nn_filt[:, band]
            # Integrations the foreground filter skipped still hold their foregrounds, so neither fit uses them, and
            # their basis is not residualized (with zero weight the filter is the identity)
            w_ee       = np.where(unfiltered_ee[:, band], 0.0, n_ee[:, band])
            w_nn       = np.where(unfiltered_nn[:, band], 0.0, n_nn[:, band])
            filt_w_ee  = np.where(unfiltered_ee[:, band], 0.0, filt_wgts_ee[:, band])
            filt_w_nn  = np.where(unfiltered_nn[:, band], 0.0, filt_wgts_nn[:, band])
            # The source model's part inside the foreground span (zero unless it is the physical-basis model)
            in_span_ee = np.zeros_like(fg_filt_ee)
            in_span_nn = np.zeros_like(fg_filt_nn)

            if RESIDUALIZED_SOURCE_BASIS:
                A_F, physical_basis, basis_info = _foreground_ranked_rm_basis(
                    freq_basis, rm_phasor, hd.freqs[band],
                    foreground_half_width, eigenval_cutoff,
                    max_foreground_fraction=SOURCE_MAX_FOREGROUND_FRACTION,
                    rcond=SOURCE_BASIS_RCOND,
                )
                n_kept = int(np.count_nonzero(basis_info["keep"]))
                if n_kept == 0:
                    print(
                        f"  WARNING: {source_name} has no physical RM modes with "
                        f"foreground fraction <= {SOURCE_MAX_FOREGROUND_FRACTION:g} "
                        f"in {hd.freqs[band][0] / 1e6:.1f}-"
                        f"{hd.freqs[band][-1] / 1e6:.1f} MHz; using a zero RM model."
                    )
                    rm_model_ee = np.zeros_like(fg_filt_ee)
                    rm_model_nn = np.zeros_like(fg_filt_nn)
                else:
                    max_fg = basis_info["foreground_fraction"][basis_info["keep"]].max()
                    print(
                        f"  {source_name}: retained {n_kept}/"
                        f"{basis_info['n_source_modes']} physical RM modes in "
                        f"{hd.freqs[band][0] / 1e6:.1f}-"
                        f"{hd.freqs[band][-1] / 1e6:.1f} MHz "
                        f"(max foreground fraction {max_fg:.3g})."
                    )

                    rm_model_ee, in_span_ee = _residualized_rm_model(
                        fg_filt_ee, w_ee, filt_w_ee, time_basis,
                        A_F, physical_basis, ridge_alpha=eigenval_cutoff,
                        physical=PHYSICAL_SOURCE_MODEL,
                    )
                    rm_model_nn, in_span_nn = _residualized_rm_model(
                        fg_filt_nn, w_nn, filt_w_nn, time_basis,
                        A_F, physical_basis, ridge_alpha=eigenval_cutoff,
                        physical=PHYSICAL_SOURCE_MODEL,
                    )
                    # The foreground filter removed the source's part inside its span along with
                    # the foregrounds, and the physical-basis model holds that part. Move it from the
                    # foreground model to the source model on unflagged cells, so fg_model + rm_model
                    # is unchanged wherever both are subtracted. (Zero for the residualized-basis model.)
                    fg_model_ee_out[t_start:t_end, band] -= np.where(flags_here_ee[:, band], 0.0, in_span_ee)
                    fg_model_nn_out[t_start:t_end, band] -= np.where(flags_here_nn[:, band], 0.0, in_span_nn)
            else:
                p_right_ee = _dpss_fit(fg_filt_ee * rm_phasor.conj(), w_ee, time_basis, freq_basis)
                p_left_ee  = _dpss_fit(fg_filt_ee * rm_phasor,        w_ee, time_basis, freq_basis)
                p_right_nn = _dpss_fit(fg_filt_nn * rm_phasor.conj(), w_nn, time_basis, freq_basis)
                p_left_nn  = _dpss_fit(fg_filt_nn * rm_phasor,        w_nn, time_basis, freq_basis)

                rm_model_ee = p_left_ee * rm_phasor.conj() + p_right_ee * rm_phasor
                rm_model_nn = p_left_nn * rm_phasor.conj() + p_right_nn * rm_phasor

            # The Faraday model's filtered response, which is what the filtered data contain
            rm_filtered_ee = rm_model_ee - in_span_ee
            rm_filtered_nn = rm_model_nn - in_span_nn

            if source_info["fit_scintillation"]:
                scint_model_ee, _, _ = dspec.fourier_filter(
                    times_msec, (fg_filt_ee - rm_filtered_ee), w_ee, **time_filter_kwargs
                )
                scint_model_nn, _, _ = dspec.fourier_filter(
                    times_msec, (fg_filt_nn - rm_filtered_nn), w_nn, **time_filter_kwargs
                )
            else:
                scint_model_ee = np.zeros_like(fg_filt_ee)
                scint_model_nn = np.zeros_like(fg_filt_nn)

            # Write into the trimmed slice of each intermediate output array
            rm_model_ee_out[t_start:t_end,    band] = rm_model_ee
            rm_model_nn_out[t_start:t_end,    band] = rm_model_nn
            rm_filt_ee_out[t_start:t_end,     band] = rm_filtered_ee
            rm_filt_nn_out[t_start:t_end,     band] = rm_filtered_nn
            scint_model_ee_out[t_start:t_end, band] = scint_model_ee
            scint_model_nn_out[t_start:t_end, band] = scint_model_nn
            fg_filt_ee_out[t_start:t_end,     band] = fg_filt_ee
            fg_filt_nn_out[t_start:t_end,     band] = fg_filt_nn
            wgts_ee_out[t_start:t_end,        band] = n_ee[:, band]
            wgts_nn_out[t_start:t_end,        band] = n_nn[:, band]
            residual_ee_out[t_start:t_end,    band] = fg_filt_ee - rm_filtered_ee - scint_model_ee
            residual_nn_out[t_start:t_end,    band] = fg_filt_nn - rm_filtered_nn - scint_model_nn

        # Allocate full-night output arrays and assign via boolean mask
        full_rm_model_ee    = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_rm_model_nn    = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_rm_filtered_ee = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_rm_filtered_nn = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_scint_model_ee = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_scint_model_nn = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_fg_filtered_ee = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_fg_filtered_nn = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_wgts_ee        = np.zeros((hd.times.shape[0], n_freqs), dtype=float)
        full_wgts_nn        = np.zeros((hd.times.shape[0], n_freqs), dtype=float)
        residual_ee         = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        residual_nn         = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_fg_model_ee    = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)
        full_fg_model_nn    = np.zeros((hd.times.shape[0], n_freqs), dtype=complex)

        full_rm_model_ee[mask]    = rm_model_ee_out
        full_rm_model_nn[mask]    = rm_model_nn_out
        full_rm_filtered_ee[mask] = rm_filt_ee_out
        full_rm_filtered_nn[mask] = rm_filt_nn_out
        full_scint_model_ee[mask] = scint_model_ee_out
        full_scint_model_nn[mask] = scint_model_nn_out
        full_fg_filtered_ee[mask] = fg_filt_ee_out
        full_fg_filtered_nn[mask] = fg_filt_nn_out
        full_wgts_ee[mask]        = wgts_ee_out
        full_wgts_nn[mask]        = wgts_nn_out
        residual_ee[mask]         = residual_ee_out
        residual_nn[mask]         = residual_nn_out
        full_fg_model_ee[mask]    = fg_model_ee_out
        full_fg_model_nn[mask]    = fg_model_nn_out

        results[source_name] = {
            "ee": {
                "fg_model":    full_fg_model_ee,
                "rm_model":    full_rm_model_ee,
                "rm_model_filtered": full_rm_filtered_ee,
                "fg_filtered": full_fg_filtered_ee,
                "scint_model": full_scint_model_ee,
                "residual":    residual_ee,
                "wgts":        full_wgts_ee,
            },
            "nn": {
                "fg_model":    full_fg_model_nn,
                "rm_model":    full_rm_model_nn,
                "rm_model_filtered": full_rm_filtered_nn,
                "fg_filtered": full_fg_filtered_nn,
                "scint_model": full_scint_model_nn,
                "residual":    residual_nn,
                "wgts":        full_wgts_nn,
            },
        }

    return results

In [ ]:
def plot_source_models_vs_data(results, band):
    '''For each modeled source, two rows of four waterfalls in noise units over the source's LST window and one band
    ('low' or 'high', split at BAND_SPLIT_FREQ): the foreground-filtered pseudo-Stokes Q data (ee - nn), where the
    Faraday-rotating model lives, and the pseudo-Stokes I data (ee + nn), where the scintillation model lives, each
    followed by the Faraday model's filtered response, the scintillation model, and the residual. The noise level comes
    from the averaged autocorrelations and the fit weights; cells without weight in both polarizations are blank.'''
    in_band = (hd_meta.freqs < BAND_SPLIT_FREQ * 1e6) if band == 'low' else (hd_meta.freqs >= BAND_SPLIT_FREQ * 1e6)
    for source_name in results:
        lst_center = best_fit_source_params[source_name]["right_ascension"] / 15 * np.pi / 12
        lst_half_width = best_fit_source_params[source_name]["lst_half_width"]
        mask = (lst_center - lst_half_width < hd_meta.lsts) & (hd_meta.lsts < lst_center + lst_half_width)
        extent = [hd_meta.freqs[0] / 1e6, hd_meta.freqs[-1] / 1e6, lst_array[mask][-1] * 12 / np.pi, lst_array[mask][0] * 12 / np.pi]

        w_ee, w_nn = results[source_name]["ee"]["wgts"], results[source_name]["nn"]["wgts"]
        with np.errstate(divide='ignore', invalid='ignore'):
            w_eff = (w_ee**-1 + w_nn**-1)**-1
            noise_est = (auto_data[auto_bl + ('ee',)] + auto_data[auto_bl + ('nn',)]) / np.sqrt(4 * df * dt * w_eff.real)
        norm = np.where((w_ee > 0) & (w_nn > 0), np.abs(noise_est), np.nan)
        used_chans = in_band & np.any((w_ee > 0) & (w_nn > 0), axis=0)
        if not np.any(used_chans):
            print(f'{source_name}: no channels with weight in the {band} band.')
            continue

        for stokes, sign in [('pQ', -1), ('pI', 1)]:
            data, rm_filtered, scint_model = [results[source_name]["ee"][component] + sign * results[source_name]["nn"][component]
                                              for component in ["fg_filtered", "rm_model_filtered", "scint_model"]]
            fig, axs = plt.subplots(1, 4, figsize=(18, 5), sharey=True, sharex=True)
            for ax, waterfall, title in zip(axs, [data, rm_filtered, scint_model, data - rm_filtered - scint_model],
                                            [f'Foreground-Filtered {stokes} Data', 'Faraday Model (Filtered)', 'Scintillation Model', 'Residual (Data - Model)']):
                im = ax.imshow(np.abs(waterfall)[mask] / norm[mask], cmap='turbo', aspect='auto', interpolation='none', vmin=0, vmax=3, extent=extent)
                ax.set_title(f'{source_name} -- {title}')
                ax.set_xlabel('Frequency (MHz)')
            axs[0].set_xlim(hd_meta.freqs[used_chans][0] / 1e6, hd_meta.freqs[used_chans][-1] / 1e6)
            axs[0].set_ylabel('LST (hours)')
            plt.tight_layout()
            plt.colorbar(im, ax=axs, label=f'|{stokes}| (SNR)', extend='max', pad=0.01)
            display(fig)
            plt.close(fig)

In [ ]:
# Sources are processed one at a time, each loaded with the prior (sidelobe)
# sources' models subtracted from the raw visibilities if its use_sidelobe_model
# is set. The polarized sources' own models are not subtracted from one another,
# so sources whose LST windows overlap are each fit with the other still present.
 
# Start with the prior (sidelobe) sources modeled above.
pol_subtraction_sources = dict(subtraction_sources)
 
avg_data_list     = {}
avg_flags_list    = {}
avg_nsamples_list = {}
avg_times_list    = {}
results           = {}
 
for source_name, source_info in best_fit_source_params.items():
    num_modeled_sources = len(pol_subtraction_sources) if source_info["use_sidelobe_model"] else 0
    print(f"\nLoading data for {source_name} "
          f"(subtracting {num_modeled_sources} previously modelled source(s))...")

    # This source's LST window
    lst_center = source_info["right_ascension"] / 15 * np.pi / 12
    lst_range  = (
        lst_center - source_info["lst_half_width"],
        lst_center + source_info["lst_half_width"],
    )
 
    _data, _flags, _nsamples, _times = load_parallel_average_over_baseline(
        load_files,
        target_source=source_info,
        subtraction_sources=pol_subtraction_sources if source_info["use_sidelobe_model"] else {},
        max_workers=MAX_WORKERS,
        lst_range=lst_range,
    )
 
    avg_data_list[source_name]     = _data
    avg_flags_list[source_name]    = _flags
    avg_nsamples_list[source_name] = _nsamples
    avg_times_list[source_name]    = _times
 
    # Guard: keep a correctly shaped zero model if all data in this window is flagged.
    mask = (lst_range[0] < hd_meta.lsts) & (hd_meta.lsts < lst_range[1])
    combined_flags = _flags["ee"] | prior_flags[mask] | _flags["nn"]
    if np.all(combined_flags):
        print(f"  {source_name}: all data flagged in DPSS fit window; using zero model.")
        results[source_name] = _zero_model_result(hd_meta)
        continue
 
    # Fit this source's model using only its own data dict so that
    # fit_pulsar_models processes exactly one source.
    print(f"  Fitting model for {source_name}...")
    _results = fit_pulsar_models(
        hd_meta,
        {source_name: _data},
        {source_name: _flags},
        {source_name: _nsamples},
        {source_name: _times},
        temporal_half_width=MODEL_FRATE_HALF_WIDTH,
        spectral_half_width=MODEL_DELAY_HALF_WIDTH * 1e-9,
        eigenval_cutoff=EIGENVAL_CUTOFF,
        foreground_half_width=FG_DELAY_HALF_WIDTH * 1e-9,
    )
    results[source_name] = _results[source_name]
 
    print(f"  Done.  {source_name} modeled.")

In [ ]:
# the averaged autocorrelations, for the noise level in the figures below
auto_hd = io.HERAData(auto_file[0])
auto_data, _, _ = auto_hd.read()
auto_bl = auto_hd.antpairs[0]
df, dt = np.median(np.diff(auto_hd.freqs)), np.median(np.diff(auto_hd.times)) * 24 * 3600

# *Figure 4: Polarized Source Model vs. Data (Low Band)*

In [ ]:
plot_source_models_vs_data(results, 'low')

# *Figure 5: Polarized Source Model vs. Data (High Band)*

In [ ]:
plot_source_models_vs_data(results, 'high')

## Save Models and Flags

In [ ]:
if SAVE_RESULTS:
    hd_out = io.HERAData(load_files[0])
    hd_out.read(polarizations=["ee", "nn"])
    hd_out.history += add_to_history  # once, outside the loops: every model file is written from this one object
    
    for source_name in best_fit_source_params:
        # Skip sources with no results entry (none expected: an all-flagged source gets an all-zero, fully flagged model)
        if source_name not in results:
            print(f"Skipping save for {source_name}: no model was fit.")
            continue
            
        for model_name in [
            "all_comps", "fg_model", "rm_model", "scint_model",
        ]:
            #for source_name, source_info in best_fit_source_params.items():
            # Build a minimal HERAData object from the first load file as a template
            fsname = source_name.replace(" ", "_").replace("-", "_")
            source_info = best_fit_source_params[source_name]
            
            # Use the antenna pair in the first file as a dummy baseline
            fake_bl = list(hd_out.antpairs)[0]

            if model_name == "all_comps":
                lst_center = best_fit_source_params[source_name]["right_ascension"] / 15 * np.pi / 12
                lst_range  = (
                    lst_center - best_fit_source_params[source_name]["lst_half_width"],
                    lst_center + best_fit_source_params[source_name]["lst_half_width"],
                )
                mask = (lst_range[0] < hd_meta.lsts) & (hd_meta.lsts < lst_range[1])
                
                ee = np.zeros(
                    results[source_name]["ee"]["rm_model"].shape,
                    dtype=complex
                )
                nn = np.zeros(
                    results[source_name]["nn"]["rm_model"].shape,
                    dtype=complex
                )
                ee[mask] = avg_data_list[source_name]["ee"]
                nn[mask] = avg_data_list[source_name]["nn"]
                dc_out = datacontainer.DataContainer({
                    fake_bl + ("ee",): ee,
                    fake_bl + ("nn",): nn,
                })
            else:
                dc_out = datacontainer.DataContainer({
                    fake_bl + ("ee",): results[source_name]["ee"][model_name],
                    fake_bl + ("nn",): results[source_name]["nn"][model_name],
                })
            flags_out = datacontainer.DataContainer({
                fake_bl + ("ee",): results[source_name]["ee"]["wgts"] < 1,
                fake_bl + ("nn",): results[source_name]["nn"]["wgts"] < 1,
            })
            nsamples_out = datacontainer.DataContainer({
                fake_bl + ("ee",): results[source_name]["ee"]["wgts"].real,
                fake_bl + ("nn",): results[source_name]["nn"]["wgts"].real,
            })
                    
            # Attach source params and processing provenance as extra keywords
            hd_out.extra_keywords.update({
                "SOURCE_RA":               source_info["right_ascension"],
                "SOURCE_DEC":              source_info["declination"],
                "SOURCE_RM":               source_info["rotation_measure"],
                "FIT_SCINTILLATION":       source_info["fit_scintillation"],
                "SCINTILLATION_TIMESCALE": source_info["scintillation_timescale"],
                "SOURCE_NAME":             source_name,
                "FG_DELAY_HALF_WIDTH":     FG_DELAY_HALF_WIDTH * 1e-9,  # in s
                "MODEL_DELAY_HALF_WIDTH":  MODEL_DELAY_HALF_WIDTH * 1e-9,  # in s
                "MODEL_FRATE_HALF_WIDTH":  MODEL_FRATE_HALF_WIDTH,  # in mHz
                "EIGENVAL_CUTOFF":         EIGENVAL_CUTOFF,
                "SOURCE_MAX_FG_FRACTION":  SOURCE_MAX_FOREGROUND_FRACTION,
                "SOURCE_BASIS_RCOND":       SOURCE_BASIS_RCOND,
                "PHYSICAL_SOURCE_MODEL":   PHYSICAL_SOURCE_MODEL,
                "FIT_PSEUDO_STOKES_U":     FIT_PSEUDO_STOKES_U,
                "PSEUDO_STOKES_U_SIGN":    PSEUDO_STOKES_U_SIGN,
                "USE_SIDELOBE_MODEL":      source_info["use_sidelobe_model"]
            })

            outfile = os.path.join(os.path.dirname(SUM_FILE),
                                   SOURCE_MODELS_FILENAME.format(JD=jdstr, source=fsname, model=model_name))

            hd_out.update(data=dc_out, flags=flags_out, nsamples=nsamples_out)
            hd_out.write_uvh5(outfile, clobber=True)
            print(f"Wrote {outfile}")

In [ ]:
# the LST windows of the polarized sources that could not be modeled on this night, flagged at every frequency (a fully
# unflagged waterfall when every source in range was modeled): the delay-filtered pI round folds them into its flags
if SAVE_RESULTS:
    uvf_out = uvf.copy()
    uvf_out.flag_array[:] = unmodeled_source_flags[:, :, np.newaxis]
    uvf_out.history += add_to_history
    uvf_out.write(FLAGS_UNMODELED_SOURCES_FILE, clobber=True)
    print(f'Wrote {FLAGS_UNMODELED_SOURCES_FILE}, with {np.sum(np.all(unmodeled_source_flags, axis=1))} integrations flagged.')

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_filters', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')